In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "paths.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "paths.py").is_file():
    raise FileNotFoundError("Could not locate the project root")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from paths import *
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "paths.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "paths.py").is_file():
    raise FileNotFoundError("Could not locate the project root")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from paths import *
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "paths.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "paths.py").is_file():
    raise FileNotFoundError("Could not locate the project root")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from paths import *
# Install packages



In [ ]:
# Imports
# Sys
from pathlib import Path
import glob
import re
import os

# Data
import numpy as np
import pandas as pd

# Spatial Data
import xarray as xr
import geopandas as gpd
from shapely.geometry import Polygon
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import rioxarray
from rasterio.features import rasterize
import netCDF4
import h5netcdf

# Visualization
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm
from graphviz import Digraph

# Stats
import scipy.stats as stats




In [ ]:
# Paths
ROOT = DATA_DIR

BRAZIL_PATH = ROOT / "raw/geospatial/administrative/BR_UF_2024/BR_UF_2024.shp"
MUN_PATH = ROOT / "raw/geospatial/administrative/BR_Municipios_2024/BR_Municipios_2024.shp"
IND_PATH = ROOT / "raw/geospatial/traditional_communities/terras_indigenas/terras_indigenas/tis_poligonaisPolygon.shp"
QUI_PATH = ROOT / "raw/geospatial/traditional_communities/quilombolas/quilombolas/quilombolas_brasil.gpkg"
AM_LEGAL_PATH = ROOT / "raw/geospatial/legal_amazon/Limites_Amazonia_Legal_2024_shp/Limites_Amazonia_Legal_2024.shp"
MUN_AM_LEGAL_PATH = ROOT / "raw/geospatial/legal_amazon/Mun_Amazonia_Legal_2024_shp/Amazonia_Legal_Municipios_2024.shp"
MUN_PATH = ROOT / "raw/geospatial/administrative/BR_Municipios_2024/BR_Municipios_2024.shp"
POP_ELDERLY = ROOT / "raw/geospatial/elderly_population/qtd_idosos.csv"
XHWI_PATH = ROOT / "raw/era5/brazil_xhwi_monthly.nc"
OUTPUT_PATH = str(RESULTS_DIR / "risk_maps")

# Variables
EPSG = 4326

# Brazil Shapefile
BRAZIL = gpd.read_file(BRAZIL_PATH).to_crs(epsg=EPSG)



## figure 2

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Ellipse, FancyArrowPatch
from matplotlib.lines import Line2D

# ============================================================
# FIGURE 2 — Heatwave hazard mapping workflow
# High-resolution PNG for Springer
# ============================================================

plt.rcParams["font.family"] = "sans-serif"

fig, ax = plt.subplots(figsize=(7.2, 5.25), dpi=600)

ax.set_xlim(0, 1)
ax.set_ylim(-0.04, 1)
ax.axis("off")

# ============================================================
# COLORS — subtle warm palette
# ============================================================

EDGE = "#A65A32"
EDGE_FINAL = "#9A4824"

FILL_PROCESS = "#F8EFE7"
FILL_TERMINAL = "#F5E5D6"
FILL_FINAL = "#F1CDAA"

TEXT = "#2F2925"

# ============================================================
# TYPOGRAPHY
# ============================================================

FONT_SIZE = 7.4
LINE_SPACING = 1.10

# ============================================================
# NODE DIMENSIONS
# ============================================================

BOX_W = 0.245
BOX_H = 0.155

ELLIPSE_W = 0.255
ELLIPSE_H = 0.160

# ============================================================
# GLOBAL ARROW STYLE
# ============================================================

ARROW_LINEWIDTH = 1.15
ARROW_MUTATION_SCALE = 9

# ============================================================
# INDIVIDUAL ARROW TUNING
# 1: n1 -> n2
# 2: n2 -> n3
# 3: n3 -> n4
# 4: n4 -> n5
# 5: n5 -> n6
# 6: n6 -> n7
# 7: n7 -> n8
# ============================================================

ARROW_PARAMS = {
    "a1": {"shrinkA": 0.2, "shrinkB": 4.},
    "a2": {"shrinkA": 0.8, "shrinkB": 4.},
    "a3": {"shrinkA": 0.0, "shrinkB": 2.7},  # orthogonal final segment
    "a4": {"shrinkA": 0.8, "shrinkB": 4.},
    "a5": {"shrinkA": 0.8, "shrinkB": 4.},
    "a6": {"shrinkA": 0.0, "shrinkB": 2.7},  # orthogonal final segment
    "a7": {"shrinkA": 0.5, "shrinkB": 0.},
}

# ============================================================
# NODE FUNCTIONS
# ============================================================

def process_box(x, y, text):
    box = FancyBboxPatch(
        (x, y),
        BOX_W,
        BOX_H,
        boxstyle="round,pad=0.010,rounding_size=0.022",
        linewidth=1.25,
        edgecolor=EDGE,
        facecolor=FILL_PROCESS,
        zorder=3
    )

    ax.add_patch(box)

    ax.text(
        x + BOX_W / 2,
        y + BOX_H / 2,
        text,
        ha="center",
        va="center",
        fontsize=FONT_SIZE,
        fontfamily="sans-serif",
        color=TEXT,
        linespacing=LINE_SPACING,
        zorder=4
    )


def terminal_node(cx, cy, text, final=False):
    ellipse = Ellipse(
        (cx, cy),
        width=ELLIPSE_W,
        height=ELLIPSE_H,
        linewidth=1.3,
        edgecolor=EDGE_FINAL if final else EDGE,
        facecolor=FILL_FINAL if final else FILL_TERMINAL,
        zorder=3
    )

    ax.add_patch(ellipse)

    ax.text(
        cx,
        cy,
        text,
        ha="center",
        va="center",
        fontsize=FONT_SIZE,
        fontfamily="sans-serif",
        color=TEXT,
        linespacing=LINE_SPACING,
        zorder=4
    )


# ============================================================
# ARROW FUNCTIONS
# ============================================================

def straight_arrow(start, end, color=EDGE, shrinkA=0.0, shrinkB=0.0):
    arrow = FancyArrowPatch(
        start,
        end,
        arrowstyle="-|>",
        mutation_scale=ARROW_MUTATION_SCALE,
        linewidth=ARROW_LINEWIDTH,
        color=color,
        shrinkA=shrinkA,
        shrinkB=shrinkB,
        capstyle="butt",
        joinstyle="miter",
        zorder=2
    )
    ax.add_patch(arrow)


def orthogonal_arrow(points, color=EDGE, final_shrinkA=0.0, final_shrinkB=0.0):
    """
    Orthogonal connector with arrowhead only on the final segment.
    The final point must correspond to the destination-node border.
    """

    for p1, p2 in zip(points[:-2], points[1:-1]):
        ax.add_line(
            Line2D(
                [p1[0], p2[0]],
                [p1[1], p2[1]],
                linewidth=ARROW_LINEWIDTH,
                color=color,
                solid_capstyle="butt",
                solid_joinstyle="miter",
                zorder=1
            )
        )

    straight_arrow(
        points[-2],
        points[-1],
        color=color,
        shrinkA=final_shrinkA,
        shrinkB=final_shrinkB
    )


# ============================================================
# NODE POSITIONS
# ============================================================

# Top row
n1_cx = 0.135
n1_cy = 0.835

n2_x = 0.375
n2_y = 0.757

n3_x = 0.715
n3_y = 0.757

# Middle row
n4_x = 0.025
n4_y = 0.430

n5_x = 0.375
n5_y = 0.430

n6_x = 0.715
n6_y = 0.430

# Bottom row
n7_x = 0.375
n7_y = 0.105

n8_cx = 0.835
n8_cy = n7_y + BOX_H / 2

# ============================================================
# DRAW NODES
# ============================================================

terminal_node(
    n1_cx,
    n1_cy,
    "Calculate the P95 in P1\nfor each grid cell"
)

process_box(
    n2_x,
    n2_y,
    "Apply the P1 threshold\nto P1, P2 and P3"
)

process_box(
    n3_x,
    n3_y,
    "Count extreme months\nfor each grid cell"
)

process_box(
    n4_x,
    n4_y,
    "Calculate the frequency\nof extremes (%)"
)

process_box(
    n5_x,
    n5_y,
    "Apply global\nnormalization"
)

process_box(
    n6_x,
    n6_y,
    "Clip upper values and\nrescale to [1–5]"
)

process_box(
    n7_x,
    n7_y,
    "Round to integer\nhierarchical classes"
)

terminal_node(
    n8_cx,
    n8_cy,
    "Generate hazard maps",
    final=True
)

# ============================================================
# PRECISE ANCHOR POINTS
# ============================================================

# n1
n1_right = (
    n1_cx + ELLIPSE_W / 2,
    n1_cy
)

# n2
n2_left = (
    n2_x,
    n2_y + BOX_H / 2
)
n2_right = (
    n2_x + BOX_W,
    n2_y + BOX_H / 2
)

# n3
n3_left = (
    n3_x,
    n3_y + BOX_H / 2
)
n3_bottom = (
    n3_x + BOX_W / 2,
    n3_y
)

# n4
n4_top = (
    n4_x + BOX_W / 2,
    n4_y + BOX_H
)
n4_right = (
    n4_x + BOX_W,
    n4_y + BOX_H / 2
)

# n5
n5_left = (
    n5_x,
    n5_y + BOX_H / 2
)
n5_right = (
    n5_x + BOX_W,
    n5_y + BOX_H / 2
)

# n6
n6_left = (
    n6_x,
    n6_y + BOX_H / 2
)
n6_bottom = (
    n6_x + BOX_W / 2,
    n6_y
)

# n7
n7_top = (
    n7_x + BOX_W / 2,
    n7_y + BOX_H
)
n7_right = (
    n7_x + BOX_W,
    n7_y + BOX_H / 2
)

# n8
n8_left = (
    n8_cx - ELLIPSE_W / 2,
    n8_cy
)

# ============================================================
# ARROWS
# ============================================================

# 1) n1 -> n2
straight_arrow(
    n1_right,
    n2_left,
    shrinkA=ARROW_PARAMS["a1"]["shrinkA"],
    shrinkB=ARROW_PARAMS["a1"]["shrinkB"]
)

# 2) n2 -> n3
straight_arrow(
    n2_right,
    n3_left,
    shrinkA=ARROW_PARAMS["a2"]["shrinkA"],
    shrinkB=ARROW_PARAMS["a2"]["shrinkB"]
)

# 3) n3 -> n4
route_y_1 = 0.665
orthogonal_arrow(
    [
        n3_bottom,
        (n3_bottom[0], route_y_1),
        (n4_top[0], route_y_1),
        n4_top
    ],
    final_shrinkA=ARROW_PARAMS["a3"]["shrinkA"],
    final_shrinkB=ARROW_PARAMS["a3"]["shrinkB"]
)

# 4) n4 -> n5
straight_arrow(
    n4_right,
    n5_left,
    shrinkA=ARROW_PARAMS["a4"]["shrinkA"],
    shrinkB=ARROW_PARAMS["a4"]["shrinkB"]
)

# 5) n5 -> n6
straight_arrow(
    n5_right,
    n6_left,
    shrinkA=ARROW_PARAMS["a5"]["shrinkA"],
    shrinkB=ARROW_PARAMS["a5"]["shrinkB"]
)

# 6) n6 -> n7
route_y_2 = 0.325
orthogonal_arrow(
    [
        n6_bottom,
        (n6_bottom[0], route_y_2),
        (n7_top[0], route_y_2),
        n7_top
    ],
    final_shrinkA=ARROW_PARAMS["a6"]["shrinkA"],
    final_shrinkB=ARROW_PARAMS["a6"]["shrinkB"]
)

# 7) n7 -> n8
straight_arrow(
    n7_right,
    n8_left,
    color=EDGE_FINAL,
    shrinkA=ARROW_PARAMS["a7"]["shrinkA"],
    shrinkB=ARROW_PARAMS["a7"]["shrinkB"]
)

# ============================================================
# SAVE
# ============================================================

plt.savefig(
    "figure2.png",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.025,
    facecolor="white"
)

plt.close()

print("figure2.png saved at 600 dpi")



## figure 3

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Ellipse, FancyArrowPatch
from matplotlib.lines import Line2D

# ============================================================
# FIGURE 3 — Vulnerability mapping workflow
# Output: figure3.png
# ============================================================

plt.rcParams["font.family"] = "sans-serif"

fig, ax = plt.subplots(figsize=(7.2, 5.25), dpi=600)

ax.set_xlim(0, 1)
ax.set_ylim(-0.04, 1)
ax.axis("off")

# ============================================================
# COLORS
# ============================================================

IND_EDGE = "#B07A1A"
IND_FILL = "#F5E3A6"

OLD_EDGE = "#8A4B5A"
OLD_FILL = "#E7C7CF"

FIN_EDGE = "#7A6A58"
FIN_FILL = "#E8DED2"

TEXT = "#1F1B18"

# ============================================================
# TYPOGRAPHY
# ============================================================

FONT_SIZE = 7.25
LINE_SPACING = 1.08

# ============================================================
# GEOMETRY
# ============================================================

BOX_PAD = 0.010
BOX_ROUND = 0.022

BOX_W_LEFT = 0.36
BOX_H_LEFT = 0.16

BOX_W_RIGHT = 0.34
BOX_H_RIGHT = 0.13

# Ellipses match the visible width of the boxes
ELLIPSE_W_LEFT = BOX_W_LEFT + 2 * BOX_PAD
ELLIPSE_W_RIGHT = BOX_W_RIGHT + 2 * BOX_PAD
ELLIPSE_W_FINAL = BOX_W_LEFT + 2 * BOX_PAD

ELLIPSE_H_TOP = 0.14
ELLIPSE_H_FINAL = 0.14

# Visible vertical gaps
LEFT_GAP = 0.065
RIGHT_GAP = 0.045

# Final merge
MERGE_X = 0.50
MERGE_Y = 0.175
FINAL_GAP = RIGHT_GAP   # same visible size as L2

# ============================================================
# ARROWS
# ============================================================

ARROW_LINEWIDTH = 1.15
ARROW_MUTATION_SCALE = 9

FLOW_INDIGENOUS = {
    "a1": {"shrinkA": 0.0, "shrinkB": 0.0},
}

FLOW_OLDER = {
    "a1": {"shrinkA": 0.0, "shrinkB": 0.0},   # L1
    "a2": {"shrinkA": 0.0, "shrinkB": 0.0},   # L2
    "a3": {"shrinkA": 0.0, "shrinkB": 0.0},   # L3
}

FLOW_FINAL = {
    "a1": {"shrinkA": 0.0, "shrinkB": 0.0},
}

# ============================================================
# HELPERS
# ============================================================

def process_box(x, y, w, h, text, edgecolor, facecolor):
    box = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle=f"round,pad={BOX_PAD},rounding_size={BOX_ROUND}",
        linewidth=1.25,
        edgecolor=edgecolor,
        facecolor=facecolor,
        zorder=3
    )
    ax.add_patch(box)

    ax.text(
        x + w / 2,
        y + h / 2,
        text,
        ha="center",
        va="center",
        fontsize=FONT_SIZE,
        color=TEXT,
        linespacing=LINE_SPACING,
        zorder=4
    )


def terminal_node(cx, cy, w, h, text, edgecolor, facecolor):
    ellipse = Ellipse(
        (cx, cy),
        width=w,
        height=h,
        linewidth=1.3,
        edgecolor=edgecolor,
        facecolor=facecolor,
        zorder=3
    )
    ax.add_patch(ellipse)

    ax.text(
        cx,
        cy,
        text,
        ha="center",
        va="center",
        fontsize=FONT_SIZE,
        color=TEXT,
        linespacing=LINE_SPACING,
        zorder=4
    )


def straight_arrow(start, end, color, shrinkA=0.0, shrinkB=0.0):
    arrow = FancyArrowPatch(
        start,
        end,
        arrowstyle="-|>",
        mutation_scale=ARROW_MUTATION_SCALE,
        linewidth=ARROW_LINEWIDTH,
        color=color,
        shrinkA=shrinkA,
        shrinkB=shrinkB,
        capstyle="butt",
        joinstyle="miter",
        zorder=2
    )
    ax.add_patch(arrow)


def polyline(points, color):
    for p1, p2 in zip(points[:-1], points[1:]):
        ax.add_line(
            Line2D(
                [p1[0], p2[0]],
                [p1[1], p2[1]],
                linewidth=ARROW_LINEWIDTH,
                color=color,
                solid_capstyle="butt",
                solid_joinstyle="miter",
                zorder=1
            )
        )


def box_bounds(x, y, w, h):
    return {
        "left": x - BOX_PAD,
        "right": x + w + BOX_PAD,
        "bottom": y - BOX_PAD,
        "top": y + h + BOX_PAD,
        "cx": x + w / 2,
        "cy": y + h / 2,
    }

# ============================================================
# POSITIONS
# ============================================================

# Left branch
n3_x = 0.05
n3_y = 0.47
n3b = box_bounds(n3_x, n3_y, BOX_W_LEFT, BOX_H_LEFT)

# Right branch
n4_x = 0.57
n4_cx = n4_x + BOX_W_RIGHT / 2

n2_cx = n4_cx
n2_cy = 0.90

# Yellow top ellipse aligned side by side with the red one
n1_cx = n3b["cx"]
n1_cy = n2_cy

ellipse2_bottom = n2_cy - ELLIPSE_H_TOP / 2
n4_y = ellipse2_bottom - RIGHT_GAP - BOX_H_RIGHT - BOX_PAD
n4b = box_bounds(n4_x, n4_y, BOX_W_RIGHT, BOX_H_RIGHT)

n5_x = n4_x
n5_y = n4b["bottom"] - RIGHT_GAP - BOX_H_RIGHT - BOX_PAD
n5b = box_bounds(n5_x, n5_y, BOX_W_RIGHT, BOX_H_RIGHT)

n6_x = n4_x
n6_y = n5b["bottom"] - RIGHT_GAP - BOX_H_RIGHT - BOX_PAD
n6b = box_bounds(n6_x, n6_y, BOX_W_RIGHT, BOX_H_RIGHT)

# Final node
n7_cx = MERGE_X
n7_cy = MERGE_Y - FINAL_GAP - ELLIPSE_H_FINAL / 2

# ============================================================
# DRAW NODES
# ============================================================

terminal_node(
    n1_cx,
    n1_cy,
    ELLIPSE_W_LEFT,
    ELLIPSE_H_TOP,
    "Identification of\nIndigenous and Quilombola\nterritories",
    IND_EDGE,
    IND_FILL
)

terminal_node(
    n2_cx,
    n2_cy,
    ELLIPSE_W_RIGHT,
    ELLIPSE_H_TOP,
    "Calculation of the\ndemographic density of the\nolder population by city",
    OLD_EDGE,
    OLD_FILL
)

process_box(
    n3_x,
    n3_y,
    BOX_W_LEFT,
    BOX_H_LEFT,
    "Assignment of value 5 to\nIndigenous and Quilombola\nterritories and value 1 to\nthe other Brazilian territories",
    IND_EDGE,
    IND_FILL
)

process_box(
    n4_x,
    n4_y,
    BOX_W_RIGHT,
    BOX_H_RIGHT,
    "Global\nnormalization",
    OLD_EDGE,
    OLD_FILL
)

process_box(
    n5_x,
    n5_y,
    BOX_W_RIGHT,
    BOX_H_RIGHT,
    "Clip upper values and\nrescale to [1–5]",
    OLD_EDGE,
    OLD_FILL
)

process_box(
    n6_x,
    n6_y,
    BOX_W_RIGHT,
    BOX_H_RIGHT,
    "Round to integer\nhierarchical classes",
    OLD_EDGE,
    OLD_FILL
)

terminal_node(
    n7_cx,
    n7_cy,
    ELLIPSE_W_FINAL,
    ELLIPSE_H_FINAL,
    "Generate vulnerability\nmaps",
    FIN_EDGE,
    FIN_FILL
)

# ============================================================
# ANCHORS
# ============================================================

# Left
n1_bottom = (n1_cx, n1_cy - ELLIPSE_H_TOP / 2)
n3_top = (n3b["cx"], n3b["top"])
n3_bottom = (n3b["cx"], n3b["bottom"])

# Right
n2_bottom = (n2_cx, n2_cy - ELLIPSE_H_TOP / 2)
n4_top = (n4b["cx"], n4b["top"])
n4_bottom = (n4b["cx"], n4b["bottom"])

n5_top = (n5b["cx"], n5b["top"])
n5_bottom = (n5b["cx"], n5b["bottom"])

n6_top = (n6b["cx"], n6b["top"])
n6_bottom = (n6b["cx"], n6b["bottom"])

# Final
merge_point = (MERGE_X, MERGE_Y)
n7_top = (n7_cx, n7_cy + ELLIPSE_H_FINAL / 2)

# ============================================================
# CONNECTORS
# ============================================================

# Left branch
straight_arrow(
    n1_bottom,
    n3_top,
    color=IND_EDGE,
    shrinkA=FLOW_INDIGENOUS["a1"]["shrinkA"],
    shrinkB=FLOW_INDIGENOUS["a1"]["shrinkB"]
)

polyline(
    [
        n3_bottom,
        (n3_bottom[0], MERGE_Y),
        merge_point
    ],
    color=IND_EDGE
)

# Right branch
straight_arrow(
    n2_bottom,
    n4_top,
    color=OLD_EDGE,
    shrinkA=FLOW_OLDER["a1"]["shrinkA"],
    shrinkB=FLOW_OLDER["a1"]["shrinkB"]
)

straight_arrow(
    n4_bottom,
    n5_top,
    color=OLD_EDGE,
    shrinkA=FLOW_OLDER["a2"]["shrinkA"],
    shrinkB=FLOW_OLDER["a2"]["shrinkB"]
)

straight_arrow(
    n5_bottom,
    n6_top,
    color=OLD_EDGE,
    shrinkA=FLOW_OLDER["a3"]["shrinkA"],
    shrinkB=FLOW_OLDER["a3"]["shrinkB"]
)

polyline(
    [
        n6_bottom,
        (n6_bottom[0], MERGE_Y),
        merge_point
    ],
    color=OLD_EDGE
)

# Merge + final
ax.plot(
    MERGE_X,
    MERGE_Y,
    marker="o",
    markersize=2.5,
    markerfacecolor=FIN_EDGE,
    markeredgecolor=FIN_EDGE,
    zorder=5
)

straight_arrow(
    merge_point,
    n7_top,
    color=FIN_EDGE,
    shrinkA=FLOW_FINAL["a1"]["shrinkA"],
    shrinkB=FLOW_FINAL["a1"]["shrinkB"]
)

# ============================================================
# SAVE
# ============================================================

plt.savefig(
    "figure3.png",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.025,
    facecolor="white"
)

plt.close()



## figure 4

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Ellipse, FancyArrowPatch
from matplotlib.lines import Line2D

# ============================================================
# FIGURE 4 — Risk mapping workflow
# Output: figure4.png
# ============================================================

plt.rcParams["font.family"] = "sans-serif"

fig, ax = plt.subplots(figsize=(7.2, 5.25), dpi=600)

# Wider margins so the side ellipses are not clipped
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(0, 1.02)
ax.axis("off")

# ============================================================
# COLORS
# ============================================================

NODE_EDGE = "#7A6A58"
NODE_FILL = "#E8DED2"
TEXT = "#1F1B18"

# ============================================================
# TYPOGRAPHY
# ============================================================

FONT_SIZE = 7.3
LINE_SPACING = 1.08

# ============================================================
# GEOMETRY
# ============================================================

BOX_PAD = 0.010
BOX_ROUND = 0.022

BOX_W = 0.46
BOX_H_SMALL = 0.11
BOX_H_LARGE = 0.13

ELLIPSE_W = BOX_W + 2 * BOX_PAD
ELLIPSE_H = 0.14

# Same visible arrow length everywhere
V_GAP = 0.065
V_GAP_1 = 0.09

# ============================================================
# ARROWS
# ============================================================

ARROW_LINEWIDTH = 1.15
ARROW_MUTATION_SCALE = 9

FLOW_VERTICAL = {
    "a1": {"shrinkA": 0.0, "shrinkB": 0.0},
    "a2": {"shrinkA": 0.0, "shrinkB": 0.0},
    "a3": {"shrinkA": 0.0, "shrinkB": 0.0},
    "a4": {"shrinkA": 0.0, "shrinkB": 0.0},
}

# ============================================================
# HELPERS
# ============================================================

def process_box(x, y, w, h, text, edgecolor, facecolor):
    box = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle=f"round,pad={BOX_PAD},rounding_size={BOX_ROUND}",
        linewidth=1.25,
        edgecolor=edgecolor,
        facecolor=facecolor,
        zorder=3
    )
    ax.add_patch(box)

    ax.text(
        x + w / 2,
        y + h / 2,
        text,
        ha="center",
        va="center",
        fontsize=FONT_SIZE,
        color=TEXT,
        linespacing=LINE_SPACING,
        zorder=4
    )


def terminal_node(cx, cy, w, h, text, edgecolor, facecolor):
    ellipse = Ellipse(
        (cx, cy),
        width=w,
        height=h,
        linewidth=1.3,
        edgecolor=edgecolor,
        facecolor=facecolor,
        zorder=3
    )
    ax.add_patch(ellipse)

    ax.text(
        cx,
        cy,
        text,
        ha="center",
        va="center",
        fontsize=FONT_SIZE,
        color=TEXT,
        linespacing=LINE_SPACING,
        zorder=4
    )


def straight_arrow(start, end, color, shrinkA=0.0, shrinkB=0.0):
    arrow = FancyArrowPatch(
        start,
        end,
        arrowstyle="-|>",
        mutation_scale=ARROW_MUTATION_SCALE,
        linewidth=ARROW_LINEWIDTH,
        color=color,
        shrinkA=shrinkA,
        shrinkB=shrinkB,
        capstyle="butt",
        joinstyle="miter",
        zorder=2
    )
    ax.add_patch(arrow)


def polyline(points, color):
    for p1, p2 in zip(points[:-1], points[1:]):
        ax.add_line(
            Line2D(
                [p1[0], p2[0]],
                [p1[1], p2[1]],
                linewidth=ARROW_LINEWIDTH,
                color=color,
                solid_capstyle="butt",
                solid_joinstyle="miter",
                zorder=1
            )
        )


def box_bounds(x, y, w, h):
    return {
        "left": x - BOX_PAD,
        "right": x + w + BOX_PAD,
        "bottom": y - BOX_PAD,
        "top": y + h + BOX_PAD,
        "cx": x + w / 2,
        "cy": y + h / 2,
    }


def ellipse_bounds(cx, cy, w, h):
    return {
        "left": cx - w / 2,
        "right": cx + w / 2,
        "bottom": cy - h / 2,
        "top": cy + h / 2,
        "cx": cx,
        "cy": cy,
    }

# ============================================================
# POSITIONS
# ============================================================

# Top ellipses
n1_cx, n1_cy = 0.20, 0.93   # Hazard Maps
n2_cx, n2_cy = 0.80, 0.93   # Vulnerability Maps

# Merge point on the horizontal connector
MERGE_X = 0.50
MERGE_Y = n1_cy

# Centered vertical workflow
center_x = 0.50 - BOX_W / 2

# Compute all y positions from the same visible gap V_GAP
n3_x = center_x
n3_top_visible = MERGE_Y - V_GAP_1
n3_y = n3_top_visible - BOX_H_SMALL - BOX_PAD
n3b = box_bounds(n3_x, n3_y, BOX_W, BOX_H_SMALL)

n4_x = center_x
n4_top_visible = n3b["bottom"] - V_GAP
n4_y = n4_top_visible - BOX_H_LARGE - BOX_PAD
n4b = box_bounds(n4_x, n4_y, BOX_W, BOX_H_LARGE)

n5_x = center_x
n5_top_visible = n4b["bottom"] - V_GAP
n5_y = n5_top_visible - BOX_H_SMALL - BOX_PAD
n5b = box_bounds(n5_x, n5_y, BOX_W, BOX_H_SMALL)

n6_cx = 0.50
n6_top_visible = n5b["bottom"] - V_GAP
n6_cy = n6_top_visible - ELLIPSE_H / 2
n6b = ellipse_bounds(n6_cx, n6_cy, ELLIPSE_W, ELLIPSE_H)

# ============================================================
# DRAW NODES
# ============================================================

terminal_node(
    n1_cx,
    n1_cy,
    ELLIPSE_W,
    ELLIPSE_H,
    "Hazard Maps",
    NODE_EDGE,
    NODE_FILL
)

terminal_node(
    n2_cx,
    n2_cy,
    ELLIPSE_W,
    ELLIPSE_H,
    "Vulnerability Maps",
    NODE_EDGE,
    NODE_FILL
)

process_box(
    n3_x,
    n3_y,
    BOX_W,
    BOX_H_SMALL,
    "Global normalization",
    NODE_EDGE,
    NODE_FILL
)

process_box(
    n4_x,
    n4_y,
    BOX_W,
    BOX_H_LARGE,
    "Clip upper values and\nrescale to [1–5]",
    NODE_EDGE,
    NODE_FILL
)

process_box(
    n5_x,
    n5_y,
    BOX_W,
    BOX_H_SMALL,
    "Round to integer\nhierarchical classes",
    NODE_EDGE,
    NODE_FILL
)

terminal_node(
    n6_cx,
    n6_cy,
    ELLIPSE_W,
    ELLIPSE_H,
    "Risk Maps",
    NODE_EDGE,
    NODE_FILL
)

# ============================================================
# ANCHORS
# ============================================================

# Top
n1_right = (n1_cx + ELLIPSE_W / 2, n1_cy)
n2_left = (n2_cx - ELLIPSE_W / 2, n2_cy)
merge_point = (MERGE_X, MERGE_Y)

# Middle boxes
n3_top = (n3b["cx"], n3b["top"])
n3_bottom = (n3b["cx"], n3b["bottom"])

n4_top = (n4b["cx"], n4b["top"])
n4_bottom = (n4b["cx"], n4b["bottom"])

n5_top = (n5b["cx"], n5b["top"])
n5_bottom = (n5b["cx"], n5b["bottom"])

# Final
n6_top = (n6b["cx"], n6b["top"])

# ============================================================
# CONNECTORS
# ============================================================

# Hazard Maps ----- | ----- Vulnerability Maps
polyline(
    [
        n1_right,
        merge_point,
        n2_left
    ],
    color=NODE_EDGE
)

ax.plot(
    MERGE_X,
    MERGE_Y,
    marker="o",
    markersize=2.5,
    markerfacecolor=NODE_EDGE,
    markeredgecolor=NODE_EDGE,
    zorder=5
)

# Equal-length vertical connectors
straight_arrow(
    merge_point,
    n3_top,
    color=NODE_EDGE,
    shrinkA=FLOW_VERTICAL["a1"]["shrinkA"],
    shrinkB=FLOW_VERTICAL["a1"]["shrinkB"]
)

straight_arrow(
    n3_bottom,
    n4_top,
    color=NODE_EDGE,
    shrinkA=FLOW_VERTICAL["a2"]["shrinkA"],
    shrinkB=FLOW_VERTICAL["a2"]["shrinkB"]
)

straight_arrow(
    n4_bottom,
    n5_top,
    color=NODE_EDGE,
    shrinkA=FLOW_VERTICAL["a3"]["shrinkA"],
    shrinkB=FLOW_VERTICAL["a3"]["shrinkB"]
)

straight_arrow(
    n5_bottom,
    n6_top,
    color=NODE_EDGE,
    shrinkA=FLOW_VERTICAL["a4"]["shrinkA"],
    shrinkB=FLOW_VERTICAL["a4"]["shrinkB"]
)

# ============================================================
# SAVE
# ============================================================

plt.savefig(
    "figure4.png",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.03,
    facecolor="white"
)

plt.close()



## figura 7

In [ ]:
# Install packages

# Imports
# Sys
from pathlib import Path
import glob
import re
import os

# Data
import numpy as np
import pandas as pd

# Spatial Data
import xarray as xr
import geopandas as gpd
from shapely.geometry import Polygon
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import rioxarray
from rasterio.features import rasterize
import netCDF4
import h5netcdf

# Visualization
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm

# Stats
import scipy.stats as stats


# Paths
ROOT = DATA_DIR

BRAZIL_PATH = ROOT / "raw/geospatial/administrative/BR_UF_2024/BR_UF_2024.shp"
MUN_PATH = ROOT / "raw/geospatial/administrative/BR_Municipios_2024/BR_Municipios_2024.shp"
IND_PATH = ROOT / "raw/geospatial/traditional_communities/terras_indigenas/terras_indigenas/tis_poligonaisPolygon.shp"
QUI_PATH = ROOT / "raw/geospatial/traditional_communities/quilombolas/quilombolas/quilombolas_brasil.gpkg"
AM_LEGAL_PATH = ROOT / "raw/geospatial/legal_amazon/Limites_Amazonia_Legal_2024_shp/Limites_Amazonia_Legal_2024.shp"
MUN_AM_LEGAL_PATH = ROOT / "raw/geospatial/legal_amazon/Mun_Amazonia_Legal_2024_shp/Amazonia_Legal_Municipios_2024.shp"
MUN_PATH = ROOT / "raw/geospatial/administrative/BR_Municipios_2024/BR_Municipios_2024.shp"
POP_ELDERLY = ROOT / "raw/geospatial/elderly_population/qtd_idosos.csv"
XHWI_PATH = ROOT / "raw/era5/brazil_xhwi_monthly.nc"
OUTPUT_PATH = str(RESULTS_DIR / "risk_maps")

# Variables
EPSG = 4326

# Brazil Shapefile
BRAZIL = gpd.read_file(BRAZIL_PATH).to_crs(epsg=EPSG)

def df_da(df, ds):
    df = df.rename(columns={'Unnamed: 0': 'time'})
    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time')

    da = xr.DataArray(
                      df[df.columns[0]].values,
                      coords=[df.index],
                      dims=["time"],
                      name=df.columns[0]
                      )

    # Latitude and Longitude
    lat = ds.latitude if 'latitude' in ds.coords else ds.lat
    lon = ds.longitude if 'longitude' in ds.coords else ds.lon

    da_expanded = da.expand_dims({"latitude": lat, "longitude": lon})

    print(df.columns[0])

    return da_expanded


def plot_map(da, period,
             output_dir,
             shapefile,
             cmap=plt.get_cmap('RdBu'),
             bounds=np.linspace(-1, 1, 21),
             label_positions=[-1.0, -0.9, -0.8, -0.7, -0.6, -0.5, -0.4, -0.3, -0.2, -0.1,
                                  0,
                                  0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
             colorbar_labels=[-1.0, -0.9, -0.8, -0.7, -0.6, -0.5, -0.4, -0.3, -0.2, -0.1,
                                  0,
                                  0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
             title_prefix="Título",
             colorbar_label="Legenda",
             filename_prefix="Gráfico"):
    """
    Plota os mapas sobre um shapefile com base em dados de um DataArray.

    Parâmetros:
    - da: Dataset ou DataArray com coordenadas latitude/longitude
    - period: str ou intervalo de tempo (usado no título e nome do arquivo)
    - output_dir: diretório para salvar a imagem
    - shapefile: GeoDataFrame com o shapefile
    - cmap: colormap (por exemplo, plt.cm.RdBu)
    - label_positions: lista de posições dos ticks da colorbar
    - colorbar_labels: lista de labels correspondentes aos ticks
    - title_prefix: título do gráfico
    - colorbar_label: legenda
    - filename_prefix: Gráfico
    """


    # Plotagem
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={"projection": ccrs.PlateCarree()})
    shapefile.plot(ax=ax, facecolor="none", edgecolor="black", zorder=10)

    [mlon, mlat] = np.meshgrid(da.longitude, da.latitude)

    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    sc = ax.scatter(mlon, mlat,
                    c=da,
                    cmap=cmap,
                    norm=norm,
                    # alpha=0.6
                    )

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    # cbar = plt.colorbar(sm, ax=ax, orientation='horizontal', fraction=0.05, pad=0.1)
    cbar = plt.colorbar(sm, ax=ax, orientation='vertical', fraction=0.05, pad=0.05, shrink=0.35)
    cbar.set_ticks(label_positions)
    cbar.set_ticklabels(colorbar_labels)
    cbar.ax.tick_params(labelsize=10)
    cbar.ax.minorticks_off() # Remover os minor ticks
    cbar.set_label(colorbar_label, fontsize=12)

    # Configurações do gráfico
    period_str = f"{period.start[:4]}–{period.stop[:4]}"
    ax.set_title(f"{title_prefix} - {period_str} period")
#, fontsize=14)
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS)
    ax.set_extent([-75, -30, -35, 6], crs=ccrs.PlateCarree())

    # Adicionar linhas de grade
    g = ax.gridlines(draw_labels=True, color='gray', alpha=0.8,
                        linestyle=':', zorder=10, x_inline=False, y_inline=False)
    g.top_labels = False
    g.right_labels = False

    g.xlabel_style = {'size': 9}
    g.ylabel_style = {'size': 9}


    filename = f"{output_dir}/{filename_prefix}_{period_str}.png"
    plt.savefig(filename, dpi=300, bbox_inches="tight")


#________________________FUNCTIONS FOR PLOT MAPS________________________________
# Functions
def count_extremes(subset, threshold, mode="above"):
        if mode == "above":
            return (subset > threshold).sum(dim="time", skipna=True)
        elif mode == "below":
            return (subset < threshold).sum(dim="time", skipna=True)
        else:
            raise ValueError("threshold_mode must be 'above' or 'below'")


def compute_var_extremes(
                         da: xr.DataArray,
                         base_period: tuple = (1950, 1974),
                         compare_periods: list = [(1975, 1998), (1999, 2023)],
                         quantile: float = 0.95,
                         threshold_mode: str = "above"  # ou "below"
                        ) -> xr.Dataset:

    #  DataArray and 'time' in datetime format
    if not np.issubdtype(da.time.dtype, np.datetime64):
        da['time'] = pd.to_datetime(da.time.values)

    # Time mask
    year = da['time'].dt.year

    masks = {"base": (year >= base_period[0]) & (year <= base_period[1])}

    for i, (start, end) in enumerate(compare_periods):
        key = f"period_{start}_{end}"
        masks[key] = (year >= start) & (year <= end)


    # Base Percentil
    base_data = da.sel(time=masks["base"])
    threshold = base_data.quantile(quantile, dim="time", skipna=True)

    # Number of Extremes
    total_months = {}
    extreme_months = {}

    for key, mask in masks.items():
        subset = da.sel(time=mask)
        total_months[key] = subset.count(dim="time")
        extreme_months[key] = count_extremes(subset, threshold, mode=threshold_mode)

    # % of Extremes per period
    probs = {
             key: (extreme_months[key] / total_months[key]) * 100
             for key in masks
            }

    # Percentual increase from base period
    increases = {}

    for key in probs:
        if key == "base":
            continue
        increases[key] = ((probs[key] - probs["base"]) / probs["base"]) * 100

    # Output Dataset
    output = xr.Dataset(
                        {
                         "threshold": threshold,
                         **{f"prob_{key}": probs[key] for key in probs},
                         **{f"n_extreme_{key}": extreme_months[key] for key in extreme_months},
                         **{f"increase_vs_base_{key}": increases[key] for key in increases}
                        }
                       )

    return output


def normalize_ds_for_plot(ds: xr.Dataset, variables: list,
                          new_min: float=1, new_max: float=5) -> xr.Dataset:
    """
    Seleciona e normaliza variáveis de um Dataset para um intervalo definido [new_min, new_max],
    usando limites baseados em IQR, mas respeitando os valores reais mínimo e máximo do conjunto.

    Parâmetros:
        ds: xarray.Dataset contendo os dados originais.
        variables: lista de variáveis (strings) a serem selecionadas e normalizadas.
        new_min: limite inferior do novo intervalo.
        new_max: limite superior do novo intervalo.

    Retorna:
        Um novo Dataset com as variáveis normalizadas.
    """
    ds_sel = ds[variables].copy()

    # Global Values
    arr = ds_sel.to_array()
    min_value = arr.min().item()
    max_value = arr.max().item()

    # outliers
    q1 = arr.quantile(0.25).item()
    q3 = arr.quantile(0.75).item()
    iqr = q3 - q1

    lower_thr = q1 - 1.5 * iqr
    upper_thr = q3 + 1.5 * iqr

    print(f"min_value: {min_value}")
    print(f"lower_thr: {lower_thr}")
    print(f"max_value: {max_value}")
    print(f"upper_thr: {upper_thr}")

    # Adjustments to avoid extrapolation out of the distribution
    lower_thr = max(lower_thr, min_value)
    upper_thr = min(upper_thr, max_value)

    # Avoiding division per zero
    if upper_thr == lower_thr:
        ds_sel[variables] = new_min
        return ds_sel

    # Limiting values for normalization
    ds_sel = ds_sel.clip(min=lower_thr, max=upper_thr)

    # Normalization for [new_min, new_max]
    ds_sel[variables] = new_min + ((ds_sel[variables] - lower_thr) / (upper_thr - lower_thr)) * (new_max - new_min)

    # Scale evaluation
    step = (upper_thr - lower_thr) / (new_max - new_min)
    scale_values = {
                    1: lower_thr,
                    2: lower_thr + step,
                    3: lower_thr + 2 * step,
                    4: lower_thr + 3 * step,
                    5: upper_thr
                    }
    print(scale_values)

    return ds_sel


def normalize_ds_c_for_plot(ds: xr.Dataset, variables: list,
                          new_min: float=1, new_max: float=5) -> xr.Dataset:
    """
    Seleciona e normaliza variáveis de um Dataset para um intervalo definido [new_min, new_max],
    usando limites baseados em IQR, mas respeitando os valores reais mínimo e máximo do conjunto.

    Parâmetros:
        ds: xarray.Dataset contendo os dados originais.
        variables: lista de variáveis (strings) a serem selecionadas e normalizadas.
        new_min: limite inferior do novo intervalo.
        new_max: limite superior do novo intervalo.

    Retorna:
        Um novo Dataset com as variáveis normalizadas.
    """
    ds_sel = ds[variables].copy()

    arr = ds_sel.to_array()
    values = arr.values.ravel()
    values = values[~np.isnan(values)]

    min_value = values.min()
    max_value = np.mean(np.sort(values)[-5:])

    ds_sel = ds_sel.clip(min=min_value, max=max_value)

    # Normalization for [new_min, new_max]
    ds_sel[variables] = new_min + ((ds_sel[variables] - min_value) / (max_value - min_value)) * (new_max - new_min)

    # Scale evaluation
    step = (max_value - min_value) / (new_max - new_min)
    scale_values = {
                    1: min_value,
                    2: min_value + step,
                    3: min_value + 2 * step,
                    4: min_value + 3 * step,
                    5: max_value
                    }
    print(scale_values)

    return ds_sel


def normalize_risk_for_plot(ds: xr.Dataset, variables: list,
                            new_min: float=1, new_max: float=5) -> xr.Dataset:
    """
    Seleciona e normaliza variáveis de um Dataset para um intervalo definido [new_min, new_max].

    Parâmetros:
        ds: xarray.Dataset contendo os dados originais.
        variables: lista de variáveis (strings) a serem selecionadas e normalizadas.
        new_min: limite inferior do novo intervalo.
        new_max: limite superior do novo intervalo.

    Retorna:
        Um novo Dataset com as variáveis normalizadas.
    """
    ds_sel = ds[variables].copy()

    # Valores globais
    max_value = 25
    min_value = 1

    # Normalização para [new_min, new_max]
    ds_sel[variables] = new_min + ((ds_sel[variables] - min_value) / (max_value - min_value)) * (new_max - new_min)

    ds_sel[variables] = ds_sel[variables].clip(min=new_min, max=new_max)

    return ds_sel


import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature


variaveis = ["threshold", "prob_base", "prob_period_1975_1998", "prob_period_1999_2023"]


def plot_info_hazard_maps(ds, variaveis, output_dir, brasil,
                          cmap_prob=plt.cm.gist_heat_r,
                          cmap_threshold=plt.cm.Oranges):
    """
    Gera e salva mapas de threshold e probabilidades de hazard.

    Parâmetros
    ----------
    ds : xr.Dataset
        Dataset com as variáveis a serem plotadas.
    variaveis : list
        Lista com os nomes das variáveis.
    output_dir : str
        Diretório de saída.
    brasil : GeoDataFrame
        Shapefile do Brasil.
    cmap_prob : matplotlib colormap
        Colormap para as probabilidades.
    cmap_threshold : matplotlib colormap
        Colormap para o threshold.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Coordenadas
    mlon, mlat = np.meshgrid(ds.longitude, ds.latitude)

    for dado in variaveis:
        fig, ax = plt.subplots(
            figsize=(10, 10),
            subplot_kw={"projection": ccrs.PlateCarree()}
        )

        # Shapefile do Brasil
        brasil.plot(ax=ax, facecolor="none", edgecolor="black", zorder=10)

        # Escolha da escala e colormap
        if dado == "threshold":
            sc = ax.scatter(
                mlon, mlat,
                c=ds[dado],
                cmap=cmap_threshold
            )
            cbar = plt.colorbar(
                sc, ax=ax, orientation="vertical",
                fraction=0.05, pad=0.05, shrink=0.35
            )
            cbar.set_label("Threshold", fontsize=12)

        else:
            sc = ax.scatter(
                mlon, mlat,
                c=ds[dado],
                cmap=cmap_prob,
                vmin=0,
                vmax=80
            )
            cbar = plt.colorbar(
                sc, ax=ax, orientation="vertical",
                fraction=0.05, pad=0.05, shrink=0.35
            )
            cbar.set_label("Probability (%)", fontsize=12)

        cbar.ax.tick_params(labelsize=10)
        cbar.ax.minorticks_off()

        # Título
        ax.set_title(dado.replace("_", " ").title())

        # Configurações do mapa
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS)
        ax.set_extent([-75, -30, -35, 6], crs=ccrs.PlateCarree())

        # Grade
        g = ax.gridlines(
            draw_labels=True,
            color="gray",
            alpha=0.8,
            linestyle=":",
            zorder=10,
            x_inline=False,
            y_inline=False
        )
        g.top_labels = False
        g.right_labels = False
        g.xlabel_style = {"size": 9}
        g.ylabel_style = {"size": 9}

        # Salvar
        filename = f"{output_dir}/{dado}.png"
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close(fig)


def plot_hazard_maps(ds, dados, periodos, output_dir, brasil,
                     cmap=plt.cm.Reds,
                     colorbar_labels=['Very low', 'low', 'Medium', 'high', 'Very high'],
                     label_positions=[1, 2, 3, 4, 5],
                     var_name='var',
                     title='Var Monthly Events Above p95'):
    """
    Gera e salva mapas de probabilidade SPI normalizada para vários períodos.

    Parâmetros:
        ds: xr.Dataset com os dados normalizados
        dados: lista de variáveis (ex: ["prob_base", ...])
        periodos: lista de strings com os nomes dos períodos
        output_dir: diretório onde os mapas serão salvos
        brasil: GeoDataFrame do shapefile do Brasil
        cmap: colormap a ser usado (padrão: Reds)
        colorbar_labels: rótulos do colorbar
        label_positions: posições dos rótulos no colorbar
    """
    # Normalização de cor
    bounds = np.arange(0.5, 6.1, 1)

    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    for dado, period in zip(dados, periodos):
        fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={"projection": ccrs.PlateCarree()})

        # Shapefile do Brasil
        brasil.plot(ax=ax, facecolor="none", edgecolor="black", zorder=10)

        # Coordenadas
        mlon, mlat = np.meshgrid(ds.longitude, ds.latitude)

        # Scatter plot
        sc = ax.scatter(
                        mlon, mlat,
                        c=ds[dado],
                        cmap=cmap,
                        norm=norm
                        )

        # Colorbar
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, orientation='vertical',
                            fraction=0.05, pad=0.05, shrink=0.35)
        cbar.set_ticks(label_positions)
        cbar.set_ticklabels(colorbar_labels)
        cbar.ax.tick_params(labelsize=10)
        cbar.ax.minorticks_off()
        cbar.set_label('Probability', fontsize=12)

        # Configurações de mapa
        ax.set_title(f"{title} - {period} period")
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS)
        ax.set_extent([-75, -30, -35, 6], crs=ccrs.PlateCarree())

        # Grade
        g = ax.gridlines(draw_labels=True, color='gray', alpha=0.8,
                         linestyle=':', zorder=10, x_inline=False, y_inline=False)
        g.top_labels = False
        g.right_labels = False
        g.xlabel_style = {'size': 9}
        g.ylabel_style = {'size': 9}

        # Salvar
        filename = f"{output_dir}/{var_name}_Prob_{period}.png"
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close(fig)


#__________________________VULNERABILITY MAPS___________________________________
def plot_vulnerability_maps(gdf1, gdf2,
                            output_dir, brasil,
                            cmap1=plt.cm.Reds, cmap2=ListedColormap(["#F7FEE7", "#FEFE85", "#FFEA00", "#FFD230", "#E0AB00"]),
                            colorbar_labels1=['Very low', 'low', 'Medium', 'high', 'Very high'],
                            colorbar_labels2=['', '', '', '', ''],
                            label_positions=[1, 2, 3, 4, 5],
                            var_name='vulnerability',
                            title='Compound Vulnerability Map'):
    """
    Gera e salva mapas de probabilidade SPI normalizada para vários períodos.

    Parâmetros:
        gdf: GeoDataFrame com os dados normalizado
        output_dir: diretório onde os mapas serão salvos
        brasil: GeoDataFrame do shapefile do Brasil
        cmap: colormap a ser usado (padrão: Reds)
        colorbar_labels: rótulos do colorbar
        label_positions: posições dos rótulos no colorbar
    """
    # Normalização de cor
    bounds = np.arange(0.5, 6.1, 1)

    norm1 = mcolors.BoundaryNorm(bounds, cmap1.N)
    norm2 = mcolors.BoundaryNorm(bounds, cmap2.N)

    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={"projection": ccrs.PlateCarree()})
    # Shapefile do Brasil
    brasil.plot(ax=ax, facecolor="none", edgecolor="black", zorder=10)

    # Scatter plot
    gdf1.plot(
              column=var_name,  # Coluna usada para colorir os polígonos
              cmap=cmap1,
              norm=norm1,
              edgecolor='black',           # (opcional) borda dos polígonos
              linewidth=0.05,               # (opcional) espessura da borda
              ax=ax                        # plota no eixo existente
              )
    gdf2.plot(
              column=var_name,  # Coluna usada para colorir os polígonos
              cmap=cmap2,
              norm=norm2,
              edgecolor='black',           # (opcional) borda dos polígonos
              linewidth=0.05,               # (opcional) espessura da borda
              ax=ax                        # plota no eixo existente
              )

    # Colorbar
    sm1 = plt.cm.ScalarMappable(cmap=cmap1, norm=norm1)
    sm1.set_array([])
    sm2 = plt.cm.ScalarMappable(cmap=cmap2, norm=norm2)
    sm2.set_array([])

    cbar1 = plt.colorbar(sm1, ax=ax, orientation='vertical',
                         fraction=0.05, pad=0.05, shrink=0.35)

    cbar1.set_ticks(label_positions)
    cbar1.set_ticklabels(colorbar_labels1)
    cbar1.ax.tick_params(labelsize=10)
    cbar1.ax.minorticks_off()
    cbar1.set_label('Probability', fontsize=12)

    cbar2 = plt.colorbar(sm2, ax=ax, orientation='vertical',
                         fraction=0.05, pad=0.05, shrink=0.35)

    cbar2.set_ticks(label_positions)
    cbar2.set_ticklabels(colorbar_labels2)
    cbar2.ax.tick_params(labelsize=10)
    cbar2.ax.minorticks_off()

    # Configurações de mapa
    ax.set_title(f"{title}")
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS)
    ax.set_extent([-75, -30, -35, 6], crs=ccrs.PlateCarree())

    # Grade
    g = ax.gridlines(draw_labels=True, color='gray', alpha=0.8,
                     linestyle=':', zorder=10, x_inline=False, y_inline=False)

    g.top_labels = False
    g.right_labels = False
    g.xlabel_style = {'size': 9}
    g.ylabel_style = {'size': 9}
    # Salvar
    filename = f"{output_dir}/{title}_Prob_{var_name}_map.png"
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close(fig)


#_______________________________RISK MAPS_______________________________________
def mask_xr(gdf, da):
    # Criar uma máscara rasterizada do shapefile com 1
    shape_mask = rasterize(
        [(geom, val) for geom, val in zip(gdf.geometry,
                                          gdf["vulnerability"])],
        out_shape=(da.sizes["latitude"], da.sizes["longitude"]),
        transform=da.rio.transform(),
        fill=0,  # Mantenho 0? Ou 1?
        dtype="float32",          # mantém decimais 1–5
    )

    # Converter para xarray com as mesmas coordenadas das ondas de calor
    mask_xr = xr.DataArray(shape_mask, coords=[da["latitude"], da["longitude"]], dims=["latitude", "longitude"])

    return mask_xr


def plot_risk_maps(ds,
                   gdf,
                   dados, periodos, output_dir, brasil,
                   cmap=plt.cm.Reds,
                   colorbar_labels=['Very low', 'low', 'Medium', 'high', 'Very high'],
                   label_positions=[1, 2, 3, 4, 5],
                   var_name='var',
                   title='Var Monthly Events Above p95'):
    """
    Gera e salva mapas de probabilidade SPI normalizada para vários períodos.

    Parâmetros:
        ds: xr.Dataset com os dados normalizados
        dados: lista de variáveis (ex: ["prob_base", ...])
        periodos: lista de strings com os nomes dos períodos
        output_dir: diretório onde os mapas serão salvos
        brasil: GeoDataFrame do shapefile do Brasil
        cmap: colormap a ser usado (padrão: Reds)
        colorbar_labels: rótulos do colorbar
        label_positions: posições dos rótulos no colorbar
    """
    # Normalização de cor
    bounds = np.arange(0.5, 6.1, 1)

    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    mask = mask_xr(gdf, ds)

    ds_mask = ds * mask

    ds_mask_norm = xr.where(ds_mask[dados] >= 1,
                            normalize_risk_for_plot(ds_mask, dados, new_min=1, new_max=5).round(),
                            ds_mask[dados])

    for dado, period in zip(dados, periodos):
        fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={"projection": ccrs.PlateCarree()})

        # Shapefile do Brasil
        brasil.plot(ax=ax, facecolor="none", edgecolor="black", zorder=10)

        # Coordenadas
        mlon, mlat = np.meshgrid(ds.longitude, ds.latitude)

        # Scatter plot
        sc = ax.scatter(
                        mlon, mlat,
                        c=ds_mask_norm[dado],
                        cmap=cmap,
                        norm=norm
                        )

        # Colorbar
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, orientation='vertical',
                            fraction=0.05, pad=0.05, shrink=0.35)
        cbar.set_ticks(label_positions)
        cbar.set_ticklabels(colorbar_labels)
        cbar.ax.tick_params(labelsize=10)
        cbar.ax.minorticks_off()
        cbar.set_label('Probability', fontsize=12)

        # Configurações de mapa
        ax.set_title(f"{title} - {period} period")
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS)
        ax.set_extent([-75, -30, -35, 6], crs=ccrs.PlateCarree())

        # Grade
        g = ax.gridlines(draw_labels=True, color='gray', alpha=0.8,
                         linestyle=':', zorder=10, x_inline=False, y_inline=False)
        g.top_labels = False
        g.right_labels = False
        g.xlabel_style = {'size': 9}
        g.ylabel_style = {'size': 9}

        # Salvar
        filename = f"{output_dir}/Risk_Map_{var_name}_Prob_{period}.png"
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close(fig)

def plot_compound_risk_maps(ds,
                            gdf1, gdf2,
                            dados, periodos, output_dir, brasil,
                            cmap1=plt.cm.Reds, cmap2=ListedColormap(["#F7FEE7", "#FEFE85", "#FFEA00", "#FFD230", "#E0AB00"]),
                            colorbar_labels1=['Very low', 'low', 'Medium', 'high', 'Very high'],
                            colorbar_labels2=['', '', '', '', ''],
                            label_positions=[1, 2, 3, 4, 5],
                            var_name='var',
                            title='Var Monthly Events Above p95'):

    """
    Gera e salva mapas de probabilidade SPI normalizada para vários períodos.

    Parâmetros:
        ds: xr.Dataset com os dados normalizados
        dados: lista de variáveis (ex: ["prob_base", ...])
        periodos: lista de strings com os nomes dos períodos
        output_dir: diretório onde os mapas serão salvos
        brasil: GeoDataFrame do shapefile do Brasil
        cmap: colormap a ser usado (padrão: Reds)
        colorbar_labels: rótulos do colorbar
        label_positions: posições dos rótulos no colorbar
    """
    # Normalização de cor
    bounds = np.arange(0.5, 6.1, 1)

    norm1 = mcolors.BoundaryNorm(bounds, cmap1.N)
    norm2 = mcolors.BoundaryNorm(bounds, cmap2.N)

    mask1 = mask_xr(gdf1, ds)
    mask2 = mask_xr(gdf2, ds)

    ds_mask1 = ds * mask1
    ds_mask2 = ds * mask2

    ds_mask_norm1 = xr.where(ds_mask1[dados] >= 1,
                            normalize_risk_for_plot(ds_mask1, dados, new_min=1, new_max=5).round(),
                            ds_mask1[dados])
    ds_mask_norm2 = xr.where(ds_mask2[dados] >= 1,
                            normalize_risk_for_plot(ds_mask2, dados, new_min=1, new_max=5).round(),
                            ds_mask2[dados])

    for dado, period in zip(dados, periodos):
        fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={"projection": ccrs.PlateCarree()})

        brasil.plot(ax=ax, facecolor="none", edgecolor="black", zorder=10)

        mlon, mlat = np.meshgrid(ds.longitude, ds.latitude)

        risk1 = ds_mask_norm1[dado].values
        risk2 = ds_mask_norm2[dado].values

        vuln1 = mask1.values
        vuln2 = mask2.values

        cond_gdf2 = (vuln2 == 5) #& (vuln1 == 1)
        #cond_black = (vuln2 == 5) & (vuln1 != 1) & (vuln1 > 0)

        # Base: risco do gdf1
        sc1 = ax.scatter(
            mlon,
            mlat,
            c=risk1,
            cmap=cmap1,
            marker="s",
            norm=norm1
        )

        # Sobreposição amarela
        if np.any(cond_gdf2):
            sc2 = ax.scatter(
                mlon[cond_gdf2],
                mlat[cond_gdf2],
                c=risk2[cond_gdf2],
                cmap=cmap2,
                marker="s",
                norm=norm2
            )

        # Sobreposição preta
#        if np.any(cond_black):
#            sc3 = ax.scatter(
#                mlon[cond_black],
#                mlat[cond_black],
#                color="black"
#            )

        # Colorbar
        sm1 = plt.cm.ScalarMappable(cmap=cmap1, norm=norm1)
        sm1.set_array([])
        cbar1 = plt.colorbar(sm1, ax=ax, orientation='vertical',
                            fraction=0.05, pad=0.05, shrink=0.35)
        cbar1.set_ticks(label_positions)
        cbar1.set_ticklabels(colorbar_labels1)
        cbar1.ax.tick_params(labelsize=10)
        cbar1.ax.minorticks_off()
        cbar1.set_label('Probability', fontsize=12)
        sm2 = plt.cm.ScalarMappable(cmap=cmap2, norm=norm2)
        sm2.set_array([])
        cbar2 = plt.colorbar(sm2, ax=ax, orientation='vertical',
                            fraction=0.05, pad=0.05, shrink=0.35)
        cbar2.set_ticks(label_positions)
        cbar2.set_ticklabels(colorbar_labels2)
        cbar2.ax.tick_params(labelsize=10)
        cbar2.ax.minorticks_off()
        cbar2.set_label('', fontsize=12)

        # Configurações de mapa
        ax.set_title(f"{title} - {period} period")
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS)
        ax.set_extent([-75, -30, -35, 6], crs=ccrs.PlateCarree())

        # Grade
        g = ax.gridlines(draw_labels=True, color='gray', alpha=0.8,
                         linestyle=':', zorder=10, x_inline=False, y_inline=False)
        g.top_labels = False
        g.right_labels = False
        g.xlabel_style = {'size': 9}
        g.ylabel_style = {'size': 9}

        # Salvar
        filename = f"{output_dir}/Risk_Map_{var_name}_Prob_{period}.png"
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close(fig)

def plot_compound_risk_maps_am(ds,
                               gdf1, gdf2,
                               dados, periodos, output_dir, brasil,
                               cmap1=plt.cm.Reds, cmap2=ListedColormap(["#F7FEE7", "#FEFE85", "#FFEA00", "#FFD230", "#E0AB00"]),
                               colorbar_labels1=['Very low', 'low', 'Medium', 'high', 'Very high'],
                               colorbar_labels2=['', '', '', '', ''],
                               label_positions=[1, 2, 3, 4, 5],
                               var_name='var',
                               title='Var Monthly Events Above p95'):

    """
    Gera e salva mapas de probabilidade SPI normalizada para vários períodos.

    Parâmetros:
        ds: xr.Dataset com os dados normalizados
        dados: lista de variáveis (ex: ["prob_base", ...])
        periodos: lista de strings com os nomes dos períodos
        output_dir: diretório onde os mapas serão salvos
        brasil: GeoDataFrame do shapefile do Brasil
        cmap: colormap a ser usado (padrão: Reds)
        colorbar_labels: rótulos do colorbar
        label_positions: posições dos rótulos no colorbar
    """
    # Normalização de cor
    bounds = np.arange(0.5, 6.1, 1)

    norm1 = mcolors.BoundaryNorm(bounds, cmap1.N)
    norm2 = mcolors.BoundaryNorm(bounds, cmap2.N)

    mask1 = mask_xr(gdf1, ds)
    mask2 = mask_xr(gdf2, ds)

    ds_mask1 = ds * mask1
    ds_mask2 = ds * mask2

    ds_mask_norm1 = xr.where(ds_mask1[dados] >= 1,
                            normalize_risk_for_plot(ds_mask1, dados, new_min=1, new_max=5).round(),
                            ds_mask1[dados])
    ds_mask_norm2 = xr.where(ds_mask2[dados] >= 1,
                            normalize_risk_for_plot(ds_mask2, dados, new_min=1, new_max=5).round(),
                            ds_mask2[dados])

    am_legal = gpd.read_file(AM_LEGAL_PATH)
    mun_am_legal = gpd.read_file(MUN_AM_LEGAL_PATH)
    #mun_gdf = gpd.read_file(MUN_PATH)

    for dado, period in zip(dados, periodos):
        fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={"projection": ccrs.PlateCarree()})

        brasil.plot(ax=ax, facecolor="none", edgecolor="black", zorder=10)
        am_legal.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=1.4, zorder=10)
        mun_am_legal.plot(ax=ax, facecolor="none", edgecolor="gray", linewidth=0.1, zorder=10)
        #mun_gdf.plot(ax=ax, facecolor="none", edgecolor="gray", linewidth=0.1, zorder=10)


        mlon, mlat = np.meshgrid(ds.longitude, ds.latitude)

        risk1 = ds_mask_norm1[dado].values
        risk2 = ds_mask_norm2[dado].values

        vuln1 = mask1.values
        vuln2 = mask2.values

        cond_gdf2 = (vuln2 == 5) #& (vuln1 == 1)
        #cond_black = (vuln2 == 5) & (vuln1 != 1) & (vuln1 > 0)

        # Base: risco do gdf1
        sc1 = ax.scatter(
            mlon,
            mlat,
            c=risk1,
            cmap=cmap1,
            marker="s",
            norm=norm1
        )

        # Sobreposição amarela
        if np.any(cond_gdf2):
            sc2 = ax.scatter(
                mlon[cond_gdf2],
                mlat[cond_gdf2],
                c=risk2[cond_gdf2],
                cmap=cmap2,
                marker="s",
                norm=norm2
            )

        # Sobreposição preta
#        if np.any(cond_black):
#            sc3 = ax.scatter(
#                mlon[cond_black],
#                mlat[cond_black],
#                color="black"
#            )

        # Colorbar
        sm1 = plt.cm.ScalarMappable(cmap=cmap1, norm=norm1)
        sm1.set_array([])
        cbar1 = plt.colorbar(sm1, ax=ax, orientation='vertical',
                            fraction=0.05, pad=0.05, shrink=0.35)
        cbar1.set_ticks(label_positions)
        cbar1.set_ticklabels(colorbar_labels1)
        cbar1.ax.tick_params(labelsize=10)
        cbar1.ax.minorticks_off()
        cbar1.set_label('Probability', fontsize=12)

        sm2 = plt.cm.ScalarMappable(cmap=cmap2, norm=norm2)
        sm2.set_array([])
        cbar2 = plt.colorbar(sm2, ax=ax, orientation='vertical',
                            fraction=0.05, pad=0.05, shrink=0.35)
        cbar2.set_ticks(label_positions)
        cbar2.set_ticklabels(colorbar_labels2)
        cbar2.ax.tick_params(labelsize=10)
        cbar2.ax.minorticks_off()
        cbar2.set_label('', fontsize=12)

        # Configurações de mapa
        ax.set_title(f"{title} - {period} period")
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.BORDERS)
        #ax.set_extent([-75, -30, -35, 6], crs=ccrs.PlateCarree())  # Brasil
        ax.set_extent([-74.5, -43, -19.5, 6], crs=ccrs.PlateCarree())  # Legal Amazon
        #ax.set_extent([-55, -38, -26, -14], crs=ccrs.PlateCarree())  # Southeast

        # Grade
        g = ax.gridlines(draw_labels=True, color='gray', alpha=0.8,
                         linestyle=':', zorder=10, x_inline=False, y_inline=False)
        g.top_labels = False
        g.right_labels = False
        g.xlabel_style = {'size': 9}
        g.ylabel_style = {'size': 9}

        # Salvar
        filename = f"{output_dir}/Risk_Map_AM_{var_name}_Prob_{period}.png"
        plt.savefig(filename, dpi=300, bbox_inches="tight")
        plt.close(fig)

# Periods
periods = [
           slice("1950-01-01", "1974-12-01"),
           slice("1975-01-01", "1998-12-01"),
           slice("1999-01-01", "2023-12-01")
           ]

# XHWI
ds_xhwi = xr.open_dataset(XHWI_PATH)
ds_xhwi = ds_xhwi.sortby("time", ascending=True)

ds_xhwi = ds_xhwi.rio.write_crs("EPSG:4326")
ds_xhwi = ds_xhwi.rio.clip(BRAZIL.geometry, BRAZIL.crs, drop=True)
#ds_xhwi_DJF_inicial = ds_xhwi.sel(time=ds_xhwi.time.dt.month.isin([12, 1, 2]))   # DJF
#ds_xhwi_MAM_inicial = ds_xhwi.sel(time=ds_xhwi.time.dt.month.isin([3, 4, 5]))    # MAM
#ds_xhwi_JJA_inicial = ds_xhwi.sel(time=ds_xhwi.time.dt.month.isin([6, 7, 8]))    # JJA
#ds_xhwi_SON_inicial = ds_xhwi.sel(time=ds_xhwi.time.dt.month.isin([9, 10, 11]))  # SON
ds_xhwi_inicial = ds_xhwi.sel(time=ds_xhwi.time.dt.month.isin([11, 12, 1, 2, 3]))  # nov -- mar
#ds_xhwi_inicial = ds_xhwi.sel(time=ds_xhwi.time.dt.month.isin([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]))  # all year


# Varibles and Periods
dados = [
         "prob_base",
         "prob_period_1975_1998",
         "prob_period_1999_2023"
        ]

periodos = [
            "1950-1974",
            "1975-1998",
            "1999-2023"
            ]

#var_xhwi = ds_xhwi_DJF_inicial['Monthly_Ind_Prod']
#var_xhwi = ds_xhwi_MAM_inicial['Monthly_Ind_Prod']
#var_xhwi = ds_xhwi_JJA_inicial['Monthly_Ind_Prod']
#var_xhwi = ds_xhwi_SON_inicial['Monthly_Ind_Prod']
var_xhwi = ds_xhwi_inicial['Monthly_Ind_Prod']

# Compute Extreme
# Heatwaves
hazard_xhwi = compute_var_extremes(
                                   da=var_xhwi,
                                   base_period=(1950, 1974),
                                   compare_periods=[(1975, 1998), (1999, 2023)],
                                   quantile=0.95,
                                   threshold_mode="above"  # or "below"
                                  )

# Normalization
# Heatwaves
hazard_xhwi_norm = normalize_ds_for_plot(
                                         hazard_xhwi,
                                         dados,
                                         new_min=1, new_max=5
                                         ).round()

da = hazard_xhwi["prob_period_1999_2023"]

idx = da.argmax(dim=("latitude", "longitude"))

lat_max = da["latitude"].isel(latitude=idx["latitude"]).item()
lon_max = da["longitude"].isel(longitude=idx["longitude"]).item()
val_max = da.isel(latitude=idx["latitude"], longitude=idx["longitude"]).item()

print("Valor máximo:", val_max)
print("Latitude:", lat_max)
print("Longitude:", lon_max)

def plot_info_all_hazard_maps(
    ds=hazard_xhwi,
    variaveis=("threshold", "prob_base", "prob_period_1975_1998", "prob_period_1999_2023"),
    output_dir=OUTPUT_PATH,
    brasil=BRAZIL,
    cmap_prob=plt.cm.gist_heat_r,
    cmap_threshold=plt.cm.Oranges,
    probability_vmin=0,
    probability_vmax=80,
    filename="plot_info_all_hazard_maps.png",
    dpi=600
):
    os.makedirs(output_dir, exist_ok=True)

    panel_labels = ["(a)", "(b)", "(c)", "(d)"]
    plot_variables = list(variaveis)

    if len(plot_variables) != 4:
        raise ValueError(
            "plot_info_all_hazard_maps expects exactly four variables: "
            "'threshold', 'prob_base', 'prob_period_1975_1998', "
            "and 'prob_period_1999_2023'."
        )

    lon = ds.longitude.values
    lat = ds.latitude.values

    fig, axes = plt.subplots(
        nrows=2,
        ncols=2,
        figsize=(10, 10),
        dpi=dpi,
        subplot_kw={"projection": ccrs.PlateCarree()}
    )

    axes = axes.ravel()

    for i, (ax, dado) in enumerate(zip(axes, plot_variables)):

        values = ds[dado].squeeze().values

        # --------------------------------------------------------------
        # PANEL (a): THRESHOLD
        # --------------------------------------------------------------
        if dado == "threshold":

            valid_values = values[np.isfinite(values)]

            if valid_values.size == 0:
                raise ValueError("No finite values were found in 'threshold'.")

            threshold_min = np.nanmin(valid_values)
            threshold_max = np.nanmax(valid_values)

            sc = ax.pcolormesh(
                lon,
                lat,
                values,
                cmap=cmap_threshold,
                vmin=threshold_min,
                vmax=threshold_max,
                shading="auto",
                edgecolors="none",
                linewidth=0,
                antialiased=False,
                rasterized=True,
                transform=ccrs.PlateCarree(),
                zorder=1
            )

            cbar = fig.colorbar(
                sc,
                ax=ax,
                orientation="vertical",
                fraction=0.040,
                pad=0.025,
                shrink=0.72
            )

            cbar.set_label(
                "Threshold",
                fontsize=8,
                labelpad=5
            )

            cbar.ax.tick_params(
                labelsize=7,
                width=0.6,
                length=2.5
            )

            cbar.outline.set_linewidth(0.6)
            cbar.ax.minorticks_off()

        # --------------------------------------------------------------
        # PANELS (b), (c), (d): EXTREME-MONTH FREQUENCY
        # --------------------------------------------------------------
        else:

            sc = ax.pcolormesh(
                lon,
                lat,
                values,
                cmap=cmap_prob,
                vmin=probability_vmin,
                vmax=probability_vmax,
                shading="auto",
                edgecolors="none",
                linewidth=0,
                antialiased=False,
                rasterized=True,
                transform=ccrs.PlateCarree(),
                zorder=1
            )

            cbar = fig.colorbar(
                sc,
                ax=ax,
                orientation="vertical",
                fraction=0.040,
                pad=0.025,
                shrink=0.72
            )

            if i != 2:
                cbar.set_label(
                    "Probability (%)",
                    fontsize=8,
                    labelpad=5
                )

            cbar.set_ticks(
                np.arange(
                    probability_vmin,
                    probability_vmax + 1,
                    10
                )
            )

            cbar.ax.tick_params(
                labelsize=7,
                width=0.6,
                length=2.5
            )

            cbar.outline.set_linewidth(0.6)
            cbar.ax.minorticks_off()

        # --------------------------------------------------------------
        # BRAZILIAN STATE BOUNDARIES
        # --------------------------------------------------------------
        brasil.plot(
            ax=ax,
            facecolor="none",
            edgecolor="black",
            linewidth=0.65,
            zorder=10
        )

        # --------------------------------------------------------------
        # COASTLINES AND INTERNATIONAL BORDERS
        # --------------------------------------------------------------
        ax.add_feature(
            cfeature.COASTLINE,
            linewidth=0.55,
            zorder=11
        )

        ax.add_feature(
            cfeature.BORDERS,
            linewidth=0.45,
            zorder=11
        )

        # --------------------------------------------------------------
        # MAP EXTENT
        # --------------------------------------------------------------
        ax.set_extent(
            [-75, -30, -35, 6],
            crs=ccrs.PlateCarree()
        )

        # --------------------------------------------------------------
        # GEOGRAPHIC GRIDLINES
        # --------------------------------------------------------------
        gl = ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            color="gray",
            alpha=0.40,
            linewidth=0.45,
            linestyle=":",
            x_inline=False,
            y_inline=False,
            zorder=9
        )

        gl.top_labels = False
        gl.right_labels = False

        gl.xlabel_style = {
            "size": 6.5
        }

        gl.ylabel_style = {
            "size": 6.5
        }

        # --------------------------------------------------------------
        # PANEL LABEL
        # --------------------------------------------------------------
        ax.text(
            0.965,
            0.965,
            panel_labels[i],
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=9,
            fontweight="bold",
            color="black",
            zorder=20
        )

    # --------------------------------------------------------------
    # PANEL SPACING
    # --------------------------------------------------------------
    fig.subplots_adjust(
        left=0.055,
        right=0.975,
        bottom=0.055,
        top=0.975,
        wspace=0.12,
        hspace=0.10
    )

    # --------------------------------------------------------------
    # SAVE
    # --------------------------------------------------------------
    output_file = os.path.join(
        output_dir,
        filename
    )

    fig.savefig(
        output_file,
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=0.04,
        facecolor="white"
    )

    plt.show()
    plt.close(fig)

    print(f"Saved: {output_file}")


plot_info_all_hazard_maps(
    ds=hazard_xhwi,
    variaveis=[
        "threshold",
        "prob_base",
        "prob_period_1975_1998",
        "prob_period_1999_2023"
    ],
output_dir=str(RESULTS_DIR),
    brasil=BRAZIL,
    cmap_prob=plt.cm.gist_heat_r,
    cmap_threshold=plt.cm.Oranges,
    probability_vmin=0,
    probability_vmax=80,
    filename="plot_info_all_hazard_maps.png",
    dpi=600
)



## figure 8

In [ ]:
# =============================================================================
# FULL 3 x 3 FIGURE — HEATWAVE HAZARD, COMPOUND VULNERABILITY, AND RISK
# November–March | P1: 1950–1974 | P2: 1975–1998 | P3: 1999–2023
# Output: PROJECT_ROOT/plot_full_maps.png
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

from pathlib import Path
import os

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd

import rioxarray
from rasterio.features import rasterize

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap

import cartopy.crs as ccrs
import cartopy.feature as cfeature


# =============================================================================
# GLOBAL PLOT SETTINGS
# =============================================================================

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans"],
    "font.size": 7.0,
    "axes.titlesize": 8.0,
    "axes.labelsize": 7.0,
    "xtick.labelsize": 5.8,
    "ytick.labelsize": 5.8,
    "figure.dpi": 600,
    "savefig.dpi": 600
})


# =============================================================================
# PATHS
# =============================================================================

ROOT = DATA_DIR
BRAZIL_PATH = (
    ROOT
    / "raw/geospatial/administrative/BR_UF_2024/BR_UF_2024.shp"
)

MUN_PATH = (
    ROOT
    / "raw/geospatial/administrative/BR_Municipios_2024/BR_Municipios_2024.shp"
)

IND_PATH = (
    ROOT
    / "raw/geospatial/traditional_communities/terras_indigenas/"
      "terras_indigenas/tis_poligonaisPolygon.shp"
)

QUI_PATH = (
    ROOT
    / "raw/geospatial/traditional_communities/quilombolas/"
      "quilombolas/quilombolas_brasil.gpkg"
)

POP_ELDERLY = (
    ROOT
    / "raw/geospatial/elderly_population/qtd_idosos.csv"
)

XHWI_PATH = XHWI_NETCDF

OUTPUT_FILE = str(RESULTS_DIR / "hazard_maps" / "plot_full_maps.png")

EPSG = 4326


# =============================================================================
# COLORS — PRESERVED FROM THE ORIGINAL SCRIPT
# =============================================================================

CMAP_RED = plt.cm.Reds.copy()

CMAP_YELLOW = ListedColormap([
    "#F7FEE7",
    "#FEFE85",
    "#FFEA00",
    "#FFD230",
    "#E0AB00"
])

# Transparent missing values are important for the raster overlays
CMAP_RED.set_bad((1, 1, 1, 0))
CMAP_YELLOW.set_bad((1, 1, 1, 0))

CLASS_BOUNDS = np.arange(0.5, 6.1, 1)

NORM_RED = mcolors.BoundaryNorm(
    CLASS_BOUNDS,
    CMAP_RED.N
)

NORM_YELLOW = mcolors.BoundaryNorm(
    CLASS_BOUNDS,
    CMAP_YELLOW.N
)

CLASS_VALUES = [1, 2, 3, 4, 5]

CLASS_LABELS = [
    "Very Low",
    "Low",
    "Medium",
    "High",
    "Very High"
]


# =============================================================================
# FUNCTIONS — HAZARD
# =============================================================================

def count_extremes(subset, threshold, mode="above"):

    if mode == "above":
        return (
            subset > threshold
        ).sum(
            dim="time",
            skipna=True
        )

    elif mode == "below":
        return (
            subset < threshold
        ).sum(
            dim="time",
            skipna=True
        )

    else:
        raise ValueError(
            "threshold_mode must be 'above' or 'below'"
        )


def compute_var_extremes(
    da,
    base_period=(1950, 1974),
    compare_periods=((1975, 1998), (1999, 2023)),
    quantile=0.95,
    threshold_mode="above"
):

    if not np.issubdtype(
        da.time.dtype,
        np.datetime64
    ):
        da["time"] = pd.to_datetime(
            da.time.values
        )

    year = da["time"].dt.year

    masks = {
        "base":
        (year >= base_period[0])
        & (year <= base_period[1])
    }

    for start, end in compare_periods:

        key = f"period_{start}_{end}"

        masks[key] = (
            (year >= start)
            & (year <= end)
        )

    # -------------------------------------------------------------------------
    # P95 calculated from P1
    # -------------------------------------------------------------------------

    base_data = da.sel(
        time=masks["base"]
    )

    threshold = base_data.quantile(
        quantile,
        dim="time",
        skipna=True
    )

    # -------------------------------------------------------------------------
    # Number of months and number of extreme months
    # -------------------------------------------------------------------------

    total_months = {}
    extreme_months = {}

    for key, mask in masks.items():

        subset = da.sel(
            time=mask
        )

        total_months[key] = subset.count(
            dim="time"
        )

        extreme_months[key] = count_extremes(
            subset,
            threshold,
            mode=threshold_mode
        )

    # -------------------------------------------------------------------------
    # Frequency of extreme months (%)
    # -------------------------------------------------------------------------

    probs = {
        key:
        (
            extreme_months[key]
            / total_months[key]
        ) * 100
        for key in masks
    }

    # -------------------------------------------------------------------------
    # Percentage increase relative to P1
    # -------------------------------------------------------------------------

    increases = {}

    for key in probs:

        if key == "base":
            continue

        increases[key] = (
            (
                probs[key]
                - probs["base"]
            )
            / probs["base"]
        ) * 100

    output = xr.Dataset({

        "threshold":
            threshold,

        **{
            f"prob_{key}":
            probs[key]
            for key in probs
        },

        **{
            f"n_extreme_{key}":
            extreme_months[key]
            for key in extreme_months
        },

        **{
            f"increase_vs_base_{key}":
            increases[key]
            for key in increases
        }
    })

    return output


def normalize_ds_for_plot(
    ds,
    variables,
    new_min=1,
    new_max=5
):

    ds_sel = ds[variables].copy()

    arr = ds_sel.to_array()

    min_value = arr.min().item()
    max_value = arr.max().item()

    q1 = arr.quantile(0.25).item()
    q3 = arr.quantile(0.75).item()

    iqr = q3 - q1

    lower_thr = q1 - 1.5 * iqr
    upper_thr = q3 + 1.5 * iqr

    lower_thr = max(
        lower_thr,
        min_value
    )

    upper_thr = min(
        upper_thr,
        max_value
    )

    if upper_thr == lower_thr:

        ds_sel[variables] = new_min

        return ds_sel

    ds_sel = ds_sel.clip(
        min=lower_thr,
        max=upper_thr
    )

    ds_sel[variables] = (
        new_min
        + (
            (
                ds_sel[variables]
                - lower_thr
            )
            / (
                upper_thr
                - lower_thr
            )
        )
        * (
            new_max
            - new_min
        )
    )

    return ds_sel


# =============================================================================
# FUNCTION — RISK NORMALIZATION
# =============================================================================

def normalize_risk_for_plot(
    ds,
    variables,
    new_min=1,
    new_max=5
):

    ds_sel = ds[variables].copy()

    min_value = 1
    max_value = 25

    ds_sel[variables] = (
        new_min
        + (
            (
                ds_sel[variables]
                - min_value
            )
            / (
                max_value
                - min_value
            )
        )
        * (
            new_max
            - new_min
        )
    )

    ds_sel[variables] = (
        ds_sel[variables]
        .clip(
            min=new_min,
            max=new_max
        )
    )

    return ds_sel


# =============================================================================
# FUNCTION — RASTERIZE VULNERABILITY TO THE XHWI GRID
# =============================================================================

def mask_xr(gdf, da):

    da = (
        da.rio
        .set_spatial_dims(
            x_dim="longitude",
            y_dim="latitude",
            inplace=False
        )
        .rio
        .write_crs(
            "EPSG:4326",
            inplace=False
        )
    )

    shape_mask = rasterize(

        [
            (geom, value)

            for geom, value
            in zip(
                gdf.geometry,
                gdf["vulnerability"]
            )

            if geom is not None
            and not geom.is_empty
        ],

        out_shape=(
            da.sizes["latitude"],
            da.sizes["longitude"]
        ),

        transform=da.rio.transform(),

        fill=0,

        dtype="float32"
    )

    return xr.DataArray(

        shape_mask,

        coords={
            "latitude":
                da["latitude"],

            "longitude":
                da["longitude"]
        },

        dims=[
            "latitude",
            "longitude"
        ]
    )


# =============================================================================
# LOAD BRAZIL AND MUNICIPAL BOUNDARIES
# =============================================================================

BRAZIL = (
    gpd.read_file(
        BRAZIL_PATH
    )
    .to_crs(
        epsg=EPSG
    )
)

mun = (
    gpd.read_file(
        MUN_PATH
    )
    .to_crs(
        epsg=EPSG
    )
)

mun["CD_MUN"] = (
    mun["CD_MUN"]
    .astype(int)
)


# =============================================================================
# LOAD XHWI
# =============================================================================

ds_xhwi = xr.open_dataset(
    XHWI_PATH
)

ds_xhwi = ds_xhwi.sortby(
    "time",
    ascending=True
)

ds_xhwi = (
    ds_xhwi.rio
    .set_spatial_dims(
        x_dim="longitude",
        y_dim="latitude",
        inplace=False
    )
    .rio
    .write_crs(
        "EPSG:4326",
        inplace=False
    )
)

ds_xhwi = ds_xhwi.rio.clip(
    BRAZIL.geometry,
    BRAZIL.crs,
    drop=True
)


# =============================================================================
# NOVEMBER–MARCH ONLY
# =============================================================================

ds_xhwi_inicial = ds_xhwi.sel(

    time=ds_xhwi.time.dt.month.isin([
        11,
        12,
        1,
        2,
        3
    ])
)

var_xhwi = (
    ds_xhwi_inicial[
        "Monthly_Ind_Prod"
    ]
)


# =============================================================================
# PERIODS
# =============================================================================

dados = [
    "prob_base",
    "prob_period_1975_1998",
    "prob_period_1999_2023"
]

periodos = [
    "1950–1974",
    "1975–1998",
    "1999–2023"
]


# =============================================================================
# COMPUTE HEATWAVE HAZARD
# =============================================================================

hazard_xhwi = compute_var_extremes(

    da=var_xhwi,

    base_period=(
        1950,
        1974
    ),

    compare_periods=[
        (
            1975,
            1998
        ),
        (
            1999,
            2023
        )
    ],

    quantile=0.95,

    threshold_mode="above"
)


# =============================================================================
# NORMALIZE HAZARD TO CLASSES 1–5
# =============================================================================

hazard_xhwi_norm = normalize_ds_for_plot(

    hazard_xhwi,

    dados,

    new_min=1,

    new_max=5

).round()

hazard_xhwi_norm = (
    hazard_xhwi_norm.rio
    .set_spatial_dims(
        x_dim="longitude",
        y_dim="latitude",
        inplace=False
    )
    .rio
    .write_crs(
        "EPSG:4326",
        inplace=False
    )
)


# =============================================================================
# ELDERLY POPULATION VULNERABILITY
# =============================================================================

df_density = pd.read_csv(
    POP_ELDERLY,
    sep=";"
)

df_density_geo = (
    df_density
    .merge(

        mun[
            [
                "CD_MUN",
                "AREA_KM2",
                "geometry"
            ]
        ],

        left_on="CD_MUN",

        right_on="CD_MUN",

        how="left"

    )
    .dropna()
)

gdf_eld_density = gpd.GeoDataFrame(

    df_density_geo,

    geometry="geometry",

    crs=f"EPSG:{EPSG}"
)

gdf_eld_density[
    "elderly_density_km2"
] = (
    gdf_eld_density["total"]
    / gdf_eld_density["AREA_KM2"]
)


# =============================================================================
# IQR-BASED NORMALIZATION OF ELDERLY POPULATION DENSITY
# =============================================================================

col = "elderly_density_km2"

q1 = (
    gdf_eld_density[col]
    .quantile(0.25)
)

q3 = (
    gdf_eld_density[col]
    .quantile(0.75)
)

iqr = q3 - q1

upper_thr = (
    q3
    + 1.5 * iqr
)

lower_real = (
    gdf_eld_density[col]
    .min()
)

gdf_eld_density[
    "elderly_density_norm"
] = (
    1
    + 4
    * (
        (
            gdf_eld_density[
                "elderly_density_km2"
            ]
            - lower_real
        )
        / (
            upper_thr
            - lower_real
        )
    )
)

gdf_eld_density.loc[
    gdf_eld_density[
        "elderly_density_norm"
    ] > 5,
    "elderly_density_norm"
] = 5

gdf_eld_density.loc[
    gdf_eld_density[
        "elderly_density_norm"
    ] < 1,
    "elderly_density_norm"
] = 1

gdf_eld_density[
    "vulnerability"
] = (
    gdf_eld_density[
        "elderly_density_norm"
    ]
)

gdf_eld_density = (
    gdf_eld_density[
        [
            "CD_MUN",
            "NM_MUN",
            "elderly_density_km2",
            "vulnerability",
            "geometry"
        ]
    ]
)


# =============================================================================
# INDIGENOUS AND QUILOMBOLA TERRITORIES
# =============================================================================

ind = (
    gpd.read_file(
        IND_PATH
    )
    .to_crs(
        epsg=EPSG
    )
)

ind[
    "vulnerability"
] = 5

ind[
    "group"
] = "Indigenous"


qui = (
    gpd.read_file(
        QUI_PATH
    )
    .to_crs(
        epsg=EPSG
    )
)

qui[
    "vulnerability"
] = 5

qui[
    "group"
] = "Quilombola"


gdf_combined = pd.concat(
    [
        ind,
        qui
    ],
    ignore_index=True
)

gdf_combined = (
    gdf_combined[
        [
            "etnia_nome",
            "nm_comunid",
            "municipio_",
            "nm_municip",
            "uf_sigla",
            "cd_uf",
            "vulnerability",
            "geometry"
        ]
    ]
)

gdf_combined = gpd.GeoDataFrame(

    gdf_combined,

    geometry="geometry",

    crs=f"EPSG:{EPSG}"
)


# =============================================================================
# RASTERIZE VULNERABILITY COMPONENTS
# =============================================================================

mask_elderly = mask_xr(
    gdf_eld_density,
    hazard_xhwi_norm
)

mask_traditional = mask_xr(
    gdf_combined,
    hazard_xhwi_norm
)


# =============================================================================
# COMPUTE RISK FOR THE THREE PERIODS
# =============================================================================

ds_mask_elderly = (
    hazard_xhwi_norm[dados]
    * mask_elderly
)

ds_mask_traditional = (
    hazard_xhwi_norm[dados]
    * mask_traditional
)


risk_elderly = xr.where(

    ds_mask_elderly >= 1,

    normalize_risk_for_plot(
        ds_mask_elderly,
        dados,
        new_min=1,
        new_max=5
    ).round(),

    ds_mask_elderly
)


risk_traditional = xr.where(

    ds_mask_traditional >= 1,

    normalize_risk_for_plot(
        ds_mask_traditional,
        dados,
        new_min=1,
        new_max=5
    ).round(),

    ds_mask_traditional
)


# =============================================================================
# MAP STYLE
# =============================================================================

MAP_EXTENT = [
    -75,
    -30,
    -35,
    6
]

PANEL_LABELS = [
    "(a)",
    "(b)",
    "(c)",
    "(d)",
    "(e)",
    "(f)",
    "(g)",
    "(h)",
    "(i)"
]


def configure_map(
    ax,
    row,
    col
):

    # State boundaries
    BRAZIL.boundary.plot(
        ax=ax,
        color="0.25",
        linewidth=0.40,
        zorder=20
    )

    # Coastline and international borders
    ax.add_feature(
        cfeature.COASTLINE,
        linewidth=0.40,
        edgecolor="0.25",
        zorder=21
    )

    ax.add_feature(
        cfeature.BORDERS,
        linewidth=0.35,
        edgecolor="0.45",
        zorder=21
    )

    ax.set_extent(
        MAP_EXTENT,
        crs=ccrs.PlateCarree()
    )

    gl = ax.gridlines(

        crs=ccrs.PlateCarree(),

        draw_labels=True,

        color="0.75",

        alpha=0.55,

        linewidth=0.30,

        linestyle=":",

        x_inline=False,

        y_inline=False,

        zorder=15
    )

    gl.top_labels = False
    gl.right_labels = False

    gl.xlabel_style = {
        "size": 5.2,
        "family": "sans-serif",
        "color": "0.35"
    }

    gl.ylabel_style = {
        "size": 5.2,
        "family": "sans-serif",
        "color": "0.35"
    }


# =============================================================================
# FINAL 3 × 3 FIGURE
# =============================================================================

def plot_full_maps():

    lon = (
        hazard_xhwi_norm[
            "longitude"
        ]
        .values
    )

    lat = (
        hazard_xhwi_norm[
            "latitude"
        ]
        .values
    )

    # Springer-compatible full-page figure width
    FIG_WIDTH = 6.85
    FIG_HEIGHT = 7.55

    fig = plt.figure(

        figsize=(
            FIG_WIDTH,
            FIG_HEIGHT
        ),

        dpi=600,

        facecolor="white"
    )

    gs = fig.add_gridspec(

        nrows=3,

        ncols=3,

        left=0.055,

        right=0.815,

        bottom=0.075,

        top=0.945,

        wspace=0.045,

        hspace=0.205
    )

    axes = np.empty(
        (
            3,
            3
        ),
        dtype=object
    )

    panel_index = 0

    # =========================================================================
    # LOOP OVER P1, P2, P3
    # =========================================================================

    for row, (
        dado,
        period
    ) in enumerate(
        zip(
            dados,
            periodos
        )
    ):

        # =====================================================================
        # COLUMN 1 — HAZARD
        # =====================================================================

        ax_hazard = fig.add_subplot(
            gs[
                row,
                0
            ],
            projection=ccrs.PlateCarree()
        )

        axes[
            row,
            0
        ] = ax_hazard

        hazard_values = (
            hazard_xhwi_norm[
                dado
            ]
            .squeeze()
            .values
        )

        hazard_values = np.ma.masked_invalid(
            hazard_values
        )

        ax_hazard.pcolormesh(

            lon,

            lat,

            hazard_values,

            cmap=CMAP_RED,

            norm=NORM_RED,

            shading="auto",

            edgecolors="none",

            linewidth=0,

            antialiased=False,

            rasterized=True,

            transform=ccrs.PlateCarree(),

            zorder=1
        )

        configure_map(
            ax_hazard,
            row,
            0
        )

        ax_hazard.text(

            0.965,
            0.965,

            PANEL_LABELS[
                panel_index
            ],

            transform=ax_hazard.transAxes,

            ha="right",

            va="top",

            fontsize=7.2,

            fontweight="bold",

            family="sans-serif",

            color="black",

            zorder=50
        )

        panel_index += 1


        # =====================================================================
        # COLUMN 2 — COMPOUND VULNERABILITY
        # =====================================================================

        ax_vulnerability = fig.add_subplot(
            gs[
                row,
                1
            ],
            projection=ccrs.PlateCarree()
        )

        axes[
            row,
            1
        ] = ax_vulnerability

        # Elderly population density
        gdf_eld_density.plot(

            column="vulnerability",

            cmap=CMAP_RED,

            norm=NORM_RED,

            edgecolor="0.35",

            linewidth=0.05,

            ax=ax_vulnerability,

            zorder=2
        )

        # Indigenous and Quilombola territories
        gdf_combined.plot(

            column="vulnerability",

            cmap=CMAP_YELLOW,

            norm=NORM_YELLOW,

            edgecolor="0.35",

            linewidth=0.06,

            ax=ax_vulnerability,

            zorder=5
        )

        configure_map(
            ax_vulnerability,
            row,
            1
        )

        ax_vulnerability.text(

            0.965,
            0.965,

            PANEL_LABELS[
                panel_index
            ],

            transform=ax_vulnerability.transAxes,

            ha="right",

            va="top",

            fontsize=7.2,

            fontweight="bold",

            family="sans-serif",

            color="black",

            zorder=50
        )

        panel_index += 1


        # =====================================================================
        # COLUMN 3 — COMPOUND RISK
        # =====================================================================

        ax_risk = fig.add_subplot(
            gs[
                row,
                2
            ],
            projection=ccrs.PlateCarree()
        )

        axes[
            row,
            2
        ] = ax_risk

        risk1 = (
            risk_elderly[
                dado
            ]
            .squeeze()
            .values
        )

        risk2 = (
            risk_traditional[
                dado
            ]
            .squeeze()
            .values
        )

        traditional_mask = (
            mask_traditional
            .values
            == 5
        )

        # ---------------------------------------------------------------------
        # Elderly-population risk
        # ---------------------------------------------------------------------

        risk1_plot = np.where(
            risk1 > 0,
            risk1,
            np.nan
        )

        risk1_plot = np.ma.masked_invalid(
            risk1_plot
        )

        ax_risk.pcolormesh(

            lon,

            lat,

            risk1_plot,

            cmap=CMAP_RED,

            norm=NORM_RED,

            shading="auto",

            edgecolors="none",

            linewidth=0,

            antialiased=False,

            rasterized=True,

            transform=ccrs.PlateCarree(),

            zorder=1
        )

        # ---------------------------------------------------------------------
        # Indigenous and Quilombola risk overlay
        # ---------------------------------------------------------------------

        risk2_plot = np.where(
            traditional_mask
            & (risk2 > 0),
            risk2,
            np.nan
        )

        risk2_plot = np.ma.masked_invalid(
            risk2_plot
        )

        ax_risk.pcolormesh(

            lon,

            lat,

            risk2_plot,

            cmap=CMAP_YELLOW,

            norm=NORM_YELLOW,

            shading="auto",

            edgecolors="none",

            linewidth=0,

            antialiased=False,

            rasterized=True,

            transform=ccrs.PlateCarree(),

            zorder=4
        )

        configure_map(
            ax_risk,
            row,
            2
        )

        ax_risk.text(

            0.965,
            0.965,

            PANEL_LABELS[
                panel_index
            ],

            transform=ax_risk.transAxes,

            ha="right",

            va="top",

            fontsize=7.2,

            fontweight="bold",

            family="sans-serif",

            color="black",

            zorder=50
        )

        panel_index += 1


    # =========================================================================
    # DRAW ONCE TO OBTAIN FINAL AXIS POSITIONS
    # =========================================================================

    fig.canvas.draw()


    # =========================================================================
    # ROW TITLES
    # =========================================================================

    for row, period in enumerate(
        periodos
    ):

        pos_left = (
            axes[
                row,
                0
            ]
            .get_position()
        )

        pos_right = (
            axes[
                row,
                2
            ]
            .get_position()
        )

        title_x = (
            pos_left.x0
            + pos_right.x1
        ) / 2

        title_y = (
            pos_left.y1
            + 0.010
        )

        fig.text(

            title_x,

            title_y,

            (
                "Heatwaves and Compound "
                f"Vulnerability: {period}"
            ),

            ha="center",

            va="bottom",

            fontsize=8.2,

            fontweight="normal",

            family="sans-serif",

            color="black"
        )


    # =========================================================================
    # TWO COLOR BARS FOR EACH ROW
    # =========================================================================

    sm_red = plt.cm.ScalarMappable(
        cmap=CMAP_RED,
        norm=NORM_RED
    )

    sm_red.set_array([])

    sm_yellow = plt.cm.ScalarMappable(
        cmap=CMAP_YELLOW,
        norm=NORM_YELLOW
    )

    sm_yellow.set_array([])


    for row in range(3):

        pos = (
            axes[
                row,
                2
            ]
            .get_position()
        )

        bar_bottom = (
            pos.y0
            + 0.16
            * pos.height
        )

        bar_height = (
            0.68
            * pos.height
        )

        # ---------------------------------------------------------------------
        # Red scale
        # ---------------------------------------------------------------------

        cax_red = fig.add_axes([
            pos.x1 + 0.010,
            bar_bottom,
            0.010,
            bar_height
        ])

        cbar_red = fig.colorbar(

            sm_red,

            cax=cax_red,

            orientation="vertical",

            boundaries=CLASS_BOUNDS,

            ticks=CLASS_VALUES,

            spacing="uniform"
        )

        cbar_red.ax.set_yticklabels(
            [
                "",
                "",
                "",
                "",
                ""
            ]
        )

        cbar_red.ax.tick_params(
            length=0,
            width=0,
            labelsize=5.5
        )

        cbar_red.outline.set_linewidth(
            0.45
        )

        # ---------------------------------------------------------------------
        # Yellow scale
        # ---------------------------------------------------------------------

        cax_yellow = fig.add_axes([
            pos.x1 + 0.026,
            bar_bottom,
            0.010,
            bar_height
        ])

        cbar_yellow = fig.colorbar(

            sm_yellow,

            cax=cax_yellow,

            orientation="vertical",

            boundaries=CLASS_BOUNDS,

            ticks=CLASS_VALUES,

            spacing="uniform"
        )

        cbar_yellow.ax.set_yticklabels(
            CLASS_LABELS
        )

        cbar_yellow.ax.tick_params(

            labelsize=5.8,

            length=0,

            width=0,

            pad=5
        )

        cbar_yellow.outline.set_linewidth(
            0.45
        )


    # =========================================================================
    # COLUMN LABELS
    # =========================================================================

    column_labels = [
        "Hazard",
        "Vulnerability",
        "Risk"
    ]

    for col, column_label in enumerate(
        column_labels
    ):

        pos = (
            axes[
                2,
                col
            ]
            .get_position()
        )

        fig.text(

            (
                pos.x0
                + pos.x1
            ) / 2,

            pos.y0 - 0.032,

            column_label,

            ha="center",

            va="top",

            fontsize=8.2,

            fontweight="normal",

            family="sans-serif",

            color="black"
        )


    # =========================================================================
    # SAVE — 600 DPI
    # =========================================================================

    os.makedirs(
        os.path.dirname(
            OUTPUT_FILE
        ),
        exist_ok=True
    )

    fig.savefig(

        OUTPUT_FILE,

        dpi=600,

        bbox_inches="tight",

        pad_inches=0.025,

        facecolor="white",

        edgecolor="none"
    )

    plt.show()

    plt.close(fig)

    print(
        f"Saved: {OUTPUT_FILE}"
    )


# =============================================================================
# RUN
# =============================================================================

plot_full_maps()

plot_vulnerability_maps(
    gdf_eld_density,
    gdf_combined,
    output_dir=str(RESULTS_DIR / "vulnerability_maps"),
    brasil=BRAZIL,
    title="Compound Vulnerability Map"
)



## figure 9

In [ ]:
# =============================================================================
# ZOOMED P3 COMPOUND RISK MAPS
# Same preprocessing used in plot_full_maps
# November–March | P3: 1999–2023
# Panel (a): Legal Amazon
# Panel (b): Southeastern Brazil
# Output:
# PROJECT_ROOT/zoom_p3risk_maps.png
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

from pathlib import Path
import os

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd

import rioxarray
from rasterio.features import rasterize

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap

import cartopy.crs as ccrs
import cartopy.feature as cfeature


# =============================================================================
# GLOBAL PLOT SETTINGS
# =============================================================================

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans"],
    "font.size": 7.0,
    "axes.titlesize": 8.0,
    "axes.labelsize": 7.0,
    "xtick.labelsize": 5.8,
    "ytick.labelsize": 5.8,
    "figure.dpi": 600,
    "savefig.dpi": 600
})


# =============================================================================
# PATHS
# =============================================================================

ROOT = DATA_DIR

BRAZIL_PATH = (
    ROOT
    / "raw/geospatial/administrative/BR_UF_2024/BR_UF_2024.shp"
)

MUN_PATH = (
    ROOT
    / "raw/geospatial/administrative/BR_Municipios_2024/BR_Municipios_2024.shp"
)

IND_PATH = (
    ROOT
    / "raw/geospatial/traditional_communities/terras_indigenas/"
      "terras_indigenas/tis_poligonaisPolygon.shp"
)

QUI_PATH = (
    ROOT
    / "raw/geospatial/traditional_communities/quilombolas/"
      "quilombolas/quilombolas_brasil.gpkg"
)

AM_LEGAL_PATH = (
    ROOT
    / "raw/geospatial/legal_amazon/Limites_Amazonia_Legal_2024_shp/"
      "Limites_Amazonia_Legal_2024.shp"
)

MUN_AM_LEGAL_PATH = (
    ROOT
    / "raw/geospatial/legal_amazon/Mun_Amazonia_Legal_2024_shp/"
      "Amazonia_Legal_Municipios_2024.shp"
)

POP_ELDERLY = (
    ROOT
    / "raw/geospatial/elderly_population/qtd_idosos.csv"
)

XHWI_PATH = XHWI_NETCDF

OUTPUT_FILE = str(RESULTS_DIR / "risk_maps" / "zoom_p3risk_maps.png")

EPSG = 4326


# =============================================================================
# COLORS — EXACTLY THE SAME AS plot_full_maps
# =============================================================================

CMAP_RED = plt.cm.Reds.copy()

CMAP_YELLOW = ListedColormap([
    "#F7FEE7",
    "#FEFE85",
    "#FFEA00",
    "#FFD230",
    "#E0AB00"
])

CMAP_RED.set_bad((1, 1, 1, 0))
CMAP_YELLOW.set_bad((1, 1, 1, 0))

CLASS_BOUNDS = np.arange(0.5, 6.1, 1)

NORM_RED = mcolors.BoundaryNorm(
    CLASS_BOUNDS,
    CMAP_RED.N
)

NORM_YELLOW = mcolors.BoundaryNorm(
    CLASS_BOUNDS,
    CMAP_YELLOW.N
)

CLASS_VALUES = [1, 2, 3, 4, 5]

CLASS_LABELS = [
    "Very Low",
    "Low",
    "Medium",
    "High",
    "Very High"
]


# =============================================================================
# FUNCTIONS — HAZARD
# =============================================================================

def count_extremes(subset, threshold, mode="above"):

    if mode == "above":
        return (
            subset > threshold
        ).sum(
            dim="time",
            skipna=True
        )

    elif mode == "below":
        return (
            subset < threshold
        ).sum(
            dim="time",
            skipna=True
        )

    else:
        raise ValueError(
            "threshold_mode must be 'above' or 'below'"
        )


def compute_var_extremes(
    da,
    base_period=(1950, 1974),
    compare_periods=((1975, 1998), (1999, 2023)),
    quantile=0.95,
    threshold_mode="above"
):

    if not np.issubdtype(
        da.time.dtype,
        np.datetime64
    ):
        da["time"] = pd.to_datetime(
            da.time.values
        )

    year = da["time"].dt.year

    masks = {
        "base":
        (year >= base_period[0])
        & (year <= base_period[1])
    }

    for start, end in compare_periods:

        key = f"period_{start}_{end}"

        masks[key] = (
            (year >= start)
            & (year <= end)
        )

    # -------------------------------------------------------------------------
    # P95 calculated from P1
    # -------------------------------------------------------------------------

    base_data = da.sel(
        time=masks["base"]
    )

    threshold = base_data.quantile(
        quantile,
        dim="time",
        skipna=True
    )

    # -------------------------------------------------------------------------
    # Number of months and number of extreme months
    # -------------------------------------------------------------------------

    total_months = {}
    extreme_months = {}

    for key, mask in masks.items():

        subset = da.sel(
            time=mask
        )

        total_months[key] = subset.count(
            dim="time"
        )

        extreme_months[key] = count_extremes(
            subset,
            threshold,
            mode=threshold_mode
        )

    # -------------------------------------------------------------------------
    # Frequency of extreme months (%)
    # -------------------------------------------------------------------------

    probs = {
        key:
        (
            extreme_months[key]
            / total_months[key]
        ) * 100
        for key in masks
    }

    # -------------------------------------------------------------------------
    # Percentage increase relative to P1
    # -------------------------------------------------------------------------

    increases = {}

    for key in probs:

        if key == "base":
            continue

        increases[key] = (
            (
                probs[key]
                - probs["base"]
            )
            / probs["base"]
        ) * 100

    output = xr.Dataset({

        "threshold":
            threshold,

        **{
            f"prob_{key}":
            probs[key]
            for key in probs
        },

        **{
            f"n_extreme_{key}":
            extreme_months[key]
            for key in extreme_months
        },

        **{
            f"increase_vs_base_{key}":
            increases[key]
            for key in increases
        }
    })

    return output


def normalize_ds_for_plot(
    ds,
    variables,
    new_min=1,
    new_max=5
):

    ds_sel = ds[variables].copy()

    arr = ds_sel.to_array()

    min_value = arr.min().item()
    max_value = arr.max().item()

    q1 = arr.quantile(0.25).item()
    q3 = arr.quantile(0.75).item()

    iqr = q3 - q1

    lower_thr = q1 - 1.5 * iqr
    upper_thr = q3 + 1.5 * iqr

    lower_thr = max(
        lower_thr,
        min_value
    )

    upper_thr = min(
        upper_thr,
        max_value
    )

    if upper_thr == lower_thr:

        ds_sel[variables] = new_min

        return ds_sel

    ds_sel = ds_sel.clip(
        min=lower_thr,
        max=upper_thr
    )

    ds_sel[variables] = (
        new_min
        + (
            (
                ds_sel[variables]
                - lower_thr
            )
            / (
                upper_thr
                - lower_thr
            )
        )
        * (
            new_max
            - new_min
        )
    )

    return ds_sel


# =============================================================================
# FUNCTION — RISK NORMALIZATION
# =============================================================================

def normalize_risk_for_plot(
    ds,
    variables,
    new_min=1,
    new_max=5
):

    ds_sel = ds[variables].copy()

    max_value = 25
    min_value = 1

    ds_sel[variables] = (
        new_min
        + (
            (
                ds_sel[variables]
                - min_value
            )
            / (
                max_value
                - min_value
            )
        )
        * (
            new_max
            - new_min
        )
    )

    ds_sel[variables] = (
        ds_sel[variables]
        .clip(
            min=new_min,
            max=new_max
        )
    )

    return ds_sel


# =============================================================================
# FUNCTION — RASTERIZE VULNERABILITY TO THE XHWI GRID
# =============================================================================

def mask_xr(gdf, da):

    da = (
        da.rio
        .set_spatial_dims(
            x_dim="longitude",
            y_dim="latitude",
            inplace=False
        )
        .rio
        .write_crs(
            "EPSG:4326",
            inplace=False
        )
    )

    shape_mask = rasterize(

        [
            (geom, value)

            for geom, value
            in zip(
                gdf.geometry,
                gdf["vulnerability"]
            )

            if geom is not None
            and not geom.is_empty
        ],

        out_shape=(
            da.sizes["latitude"],
            da.sizes["longitude"]
        ),

        transform=da.rio.transform(),

        fill=0,

        dtype="float32"
    )

    return xr.DataArray(

        shape_mask,

        coords={
            "latitude":
                da["latitude"],

            "longitude":
                da["longitude"]
        },

        dims=[
            "latitude",
            "longitude"
        ]
    )


# =============================================================================
# LOAD BRAZIL AND MUNICIPAL BOUNDARIES
# =============================================================================

BRAZIL = (
    gpd.read_file(
        BRAZIL_PATH
    )
    .to_crs(
        epsg=EPSG
    )
)

mun = (
    gpd.read_file(
        MUN_PATH
    )
    .to_crs(
        epsg=EPSG
    )
)

mun["CD_MUN"] = (
    mun["CD_MUN"]
    .astype(int)
)


# =============================================================================
# LOAD LEGAL AMAZON BOUNDARIES
# =============================================================================

am_legal = (
    gpd.read_file(
        AM_LEGAL_PATH
    )
    .to_crs(
        epsg=EPSG
    )
)

mun_am_legal = (
    gpd.read_file(
        MUN_AM_LEGAL_PATH
    )
    .to_crs(
        epsg=EPSG
    )
)


# =============================================================================
# LOAD XHWI
# =============================================================================

ds_xhwi = xr.open_dataset(
    XHWI_PATH
)

ds_xhwi = ds_xhwi.sortby(
    "time",
    ascending=True
)

ds_xhwi = (
    ds_xhwi.rio
    .set_spatial_dims(
        x_dim="longitude",
        y_dim="latitude",
        inplace=False
    )
    .rio
    .write_crs(
        "EPSG:4326",
        inplace=False
    )
)

ds_xhwi = ds_xhwi.rio.clip(
    BRAZIL.geometry,
    BRAZIL.crs,
    drop=True
)


# =============================================================================
# NOVEMBER–MARCH ONLY
# =============================================================================

ds_xhwi_inicial = ds_xhwi.sel(

    time=ds_xhwi.time.dt.month.isin([
        11,
        12,
        1,
        2,
        3
    ])
)

var_xhwi = (
    ds_xhwi_inicial[
        "Monthly_Ind_Prod"
    ]
)


# =============================================================================
# PERIODS
# =============================================================================

dados = [
    "prob_base",
    "prob_period_1975_1998",
    "prob_period_1999_2023"
]

periodos = [
    "1950–1974",
    "1975–1998",
    "1999–2023"
]

P3_VAR = "prob_period_1999_2023"


# =============================================================================
# COMPUTE HEATWAVE HAZARD
# =============================================================================

hazard_xhwi = compute_var_extremes(

    da=var_xhwi,

    base_period=(
        1950,
        1974
    ),

    compare_periods=[
        (
            1975,
            1998
        ),
        (
            1999,
            2023
        )
    ],

    quantile=0.95,

    threshold_mode="above"
)


# =============================================================================
# NORMALIZE HAZARD TO CLASSES 1–5
# =============================================================================

hazard_xhwi_norm = normalize_ds_for_plot(

    hazard_xhwi,

    dados,

    new_min=1,

    new_max=5

).round()

hazard_xhwi_norm = (
    hazard_xhwi_norm.rio
    .set_spatial_dims(
        x_dim="longitude",
        y_dim="latitude",
        inplace=False
    )
    .rio
    .write_crs(
        "EPSG:4326",
        inplace=False
    )
)


# =============================================================================
# ELDERLY POPULATION VULNERABILITY
# =============================================================================

df_density = pd.read_csv(
    POP_ELDERLY,
    sep=";"
)

df_density_geo = (
    df_density
    .merge(

        mun[
            [
                "CD_MUN",
                "AREA_KM2",
                "geometry"
            ]
        ],

        left_on="CD_MUN",

        right_on="CD_MUN",

        how="left"

    )
    .dropna()
)

gdf_eld_density = gpd.GeoDataFrame(

    df_density_geo,

    geometry="geometry",

    crs=f"EPSG:{EPSG}"
)

gdf_eld_density[
    "elderly_density_km2"
] = (
    gdf_eld_density["total"]
    / gdf_eld_density["AREA_KM2"]
)


# =============================================================================
# IQR-BASED NORMALIZATION OF ELDERLY POPULATION DENSITY
# =============================================================================

col = "elderly_density_km2"

q1 = (
    gdf_eld_density[col]
    .quantile(0.25)
)

q3 = (
    gdf_eld_density[col]
    .quantile(0.75)
)

iqr = q3 - q1

upper_thr = (
    q3
    + 1.5 * iqr
)

lower_real = (
    gdf_eld_density[col]
    .min()
)

gdf_eld_density[
    "elderly_density_norm"
] = (
    1
    + 4
    * (
        (
            gdf_eld_density[
                "elderly_density_km2"
            ]
            - lower_real
        )
        / (
            upper_thr
            - lower_real
        )
    )
)

gdf_eld_density.loc[
    gdf_eld_density[
        "elderly_density_norm"
    ] > 5,
    "elderly_density_norm"
] = 5

gdf_eld_density.loc[
    gdf_eld_density[
        "elderly_density_norm"
    ] < 1,
    "elderly_density_norm"
] = 1

gdf_eld_density[
    "vulnerability"
] = (
    gdf_eld_density[
        "elderly_density_norm"
    ]
)

gdf_eld_density = (
    gdf_eld_density[
        [
            "CD_MUN",
            "NM_MUN",
            "elderly_density_km2",
            "vulnerability",
            "geometry"
        ]
    ]
)


# =============================================================================
# INDIGENOUS AND QUILOMBOLA TERRITORIES
# =============================================================================

ind = (
    gpd.read_file(
        IND_PATH
    )
    .to_crs(
        epsg=EPSG
    )
)

ind[
    "vulnerability"
] = 5

ind[
    "group"
] = "Indigenous"


qui = (
    gpd.read_file(
        QUI_PATH
    )
    .to_crs(
        epsg=EPSG
    )
)

qui[
    "vulnerability"
] = 5

qui[
    "group"
] = "Quilombola"


gdf_combined = pd.concat(
    [
        ind,
        qui
    ],
    ignore_index=True
)

gdf_combined = (
    gdf_combined[
        [
            "etnia_nome",
            "nm_comunid",
            "municipio_",
            "nm_municip",
            "uf_sigla",
            "cd_uf",
            "vulnerability",
            "geometry"
        ]
    ]
)

gdf_combined = gpd.GeoDataFrame(

    gdf_combined,

    geometry="geometry",

    crs=f"EPSG:{EPSG}"
)


# =============================================================================
# RASTERIZE VULNERABILITY COMPONENTS
# =============================================================================

mask_elderly = mask_xr(
    gdf_eld_density,
    hazard_xhwi_norm
)

mask_traditional = mask_xr(
    gdf_combined,
    hazard_xhwi_norm
)


# =============================================================================
# COMPUTE RISK FOR THE THREE PERIODS
# EXACTLY THE SAME PREPROCESSING USED IN plot_full_maps
# =============================================================================

ds_mask_elderly = (
    hazard_xhwi_norm[dados]
    * mask_elderly
)

ds_mask_traditional = (
    hazard_xhwi_norm[dados]
    * mask_traditional
)


risk_elderly = xr.where(

    ds_mask_elderly >= 1,

    normalize_risk_for_plot(
        ds_mask_elderly,
        dados,
        new_min=1,
        new_max=5
    ).round(),

    ds_mask_elderly
)


risk_traditional = xr.where(

    ds_mask_traditional >= 1,

    normalize_risk_for_plot(
        ds_mask_traditional,
        dados,
        new_min=1,
        new_max=5
    ).round(),

    ds_mask_traditional
)


# =============================================================================
# EXTRACT P3 RISK ONLY
# =============================================================================

risk1 = (
    risk_elderly[
        P3_VAR
    ]
    .squeeze()
    .values
)

risk2 = (
    risk_traditional[
        P3_VAR
    ]
    .squeeze()
    .values
)

traditional_mask = (
    mask_traditional
    .values
    == 5
)

risk1_plot = np.where(
    risk1 > 0,
    risk1,
    np.nan
)

risk1_plot = np.ma.masked_invalid(
    risk1_plot
)

risk2_plot = np.where(
    traditional_mask
    & (risk2 > 0),
    risk2,
    np.nan
)

risk2_plot = np.ma.masked_invalid(
    risk2_plot
)


# =============================================================================
# COORDINATES
# =============================================================================

lon = (
    hazard_xhwi_norm[
        "longitude"
    ]
    .values
)

lat = (
    hazard_xhwi_norm[
        "latitude"
    ]
    .values
)


# =============================================================================
# ZOOM EXTENTS
# SAME EXTENTS DEFINED IN plot_compound_risk_maps_am
# =============================================================================

AMAZON_EXTENT = [
    -74.5,
    -43.0,
    -19.5,
    6.0
]

SOUTHEAST_EXTENT = [
    -55.0,
    -38.0,
    -26.0,
    -14.0
]


# =============================================================================
# MAP CONFIGURATION
# =============================================================================

def configure_zoom_map(
    ax,
    extent
):

    BRAZIL.boundary.plot(
        ax=ax,
        color="0.25",
        linewidth=0.45,
        zorder=20
    )

    ax.add_feature(
        cfeature.COASTLINE,
        linewidth=0.42,
        edgecolor="0.25",
        zorder=21
    )

    ax.add_feature(
        cfeature.BORDERS,
        linewidth=0.35,
        edgecolor="0.45",
        zorder=21
    )

    ax.set_extent(
        extent,
        crs=ccrs.PlateCarree()
    )

    gl = ax.gridlines(

        crs=ccrs.PlateCarree(),

        draw_labels=True,

        color="0.75",

        alpha=0.55,

        linewidth=0.30,

        linestyle=":",

        x_inline=False,

        y_inline=False,

        zorder=15
    )

    gl.top_labels = False
    gl.right_labels = False

    gl.xlabel_style = {
        "size": 5.3,
        "family": "sans-serif",
        "color": "0.35"
    }

    gl.ylabel_style = {
        "size": 5.3,
        "family": "sans-serif",
        "color": "0.35"
    }


# =============================================================================
# FINAL TWO-PANEL P3 RISK FIGURE
# =============================================================================

def plot_zoom_p3risk_maps():

    # -------------------------------------------------------------------------
    # Springer full-width figure
    # -------------------------------------------------------------------------

    FIG_WIDTH = 6.85
    FIG_HEIGHT = 3.65

    # -------------------------------------------------------------------------
    # Adjust the Southeast extent so that both panels have the same
    # geographic aspect ratio, preserving the map projection without
    # stretching either panel.
    # -------------------------------------------------------------------------

    amazon_width = AMAZON_EXTENT[1] - AMAZON_EXTENT[0]
    amazon_height = AMAZON_EXTENT[3] - AMAZON_EXTENT[2]
    target_aspect = amazon_width / amazon_height

    southeast_width = SOUTHEAST_EXTENT[1] - SOUTHEAST_EXTENT[0]
    southeast_center_lat = (
        SOUTHEAST_EXTENT[2]
        + SOUTHEAST_EXTENT[3]
    ) / 2

    southeast_height = southeast_width / target_aspect

    SOUTHEAST_EXTENT_EQUAL = [
        SOUTHEAST_EXTENT[0],
        SOUTHEAST_EXTENT[1],
        southeast_center_lat - southeast_height / 2,
        southeast_center_lat + southeast_height / 2
    ]

    # -------------------------------------------------------------------------
    # Figure
    # -------------------------------------------------------------------------

    fig = plt.figure(
        figsize=(FIG_WIDTH, FIG_HEIGHT),
        dpi=600,
        facecolor="white"
    )

    gs = fig.add_gridspec(
        nrows=1,
        ncols=2,
        left=0.055,
        right=0.975,
        bottom=0.255,
        top=0.970,
        wspace=0.16
    )


    # =========================================================================
    # PANEL (a) — P3 RISK | LEGAL AMAZON
    # =========================================================================

    ax1 = fig.add_subplot(
        gs[0, 0],
        projection=ccrs.PlateCarree()
    )

    ax1.pcolormesh(
        lon,
        lat,
        risk1_plot,
        cmap=CMAP_RED,
        norm=NORM_RED,
        shading="auto",
        edgecolors="none",
        linewidth=0,
        antialiased=False,
        rasterized=True,
        transform=ccrs.PlateCarree(),
        zorder=1
    )

    ax1.pcolormesh(
        lon,
        lat,
        risk2_plot,
        cmap=CMAP_YELLOW,
        norm=NORM_YELLOW,
        shading="auto",
        edgecolors="none",
        linewidth=0,
        antialiased=False,
        rasterized=True,
        transform=ccrs.PlateCarree(),
        zorder=4
    )

    mun_am_legal.boundary.plot(
        ax=ax1,
        facecolor="none",
        edgecolor="0.55",
        linewidth=0.10,
        zorder=12
    )

    am_legal.boundary.plot(
        ax=ax1,
        facecolor="none",
        edgecolor="red",
        linewidth=1.10,
        zorder=30
    )

    configure_zoom_map(
        ax1,
        AMAZON_EXTENT
    )

    ax1.text(
        0.975,
        0.965,
        "(a)",
        transform=ax1.transAxes,
        ha="right",
        va="top",
        fontsize=7.4,
        fontweight="bold",
        family="sans-serif",
        color="black",
        zorder=50
    )


    # =========================================================================
    # PANEL (b) — SAME P3 RISK | SOUTHEASTERN BRAZIL
    # =========================================================================

    ax2 = fig.add_subplot(
        gs[0, 1],
        projection=ccrs.PlateCarree()
    )

    ax2.pcolormesh(
        lon,
        lat,
        risk1_plot,
        cmap=CMAP_RED,
        norm=NORM_RED,
        shading="auto",
        edgecolors="none",
        linewidth=0,
        antialiased=False,
        rasterized=True,
        transform=ccrs.PlateCarree(),
        zorder=1
    )

    ax2.pcolormesh(
        lon,
        lat,
        risk2_plot,
        cmap=CMAP_YELLOW,
        norm=NORM_YELLOW,
        shading="auto",
        edgecolors="none",
        linewidth=0,
        antialiased=False,
        rasterized=True,
        transform=ccrs.PlateCarree(),
        zorder=4
    )

    mun.boundary.plot(
        ax=ax2,
        facecolor="none",
        edgecolor="0.55",
        linewidth=0.08,
        zorder=12
    )

    configure_zoom_map(
        ax2,
        SOUTHEAST_EXTENT_EQUAL
    )

    # (b) kept at the lower-right corner
    ax2.text(
        0.975,
        0.055,
        "(b)",
        transform=ax2.transAxes,
        ha="right",
        va="bottom",
        fontsize=7.4,
        fontweight="bold",
        family="sans-serif",
        color="black",
        zorder=50
    )


    # =========================================================================
    # SHARED COLOR SCALES
    # =========================================================================

    sm_yellow = plt.cm.ScalarMappable(
        cmap=CMAP_YELLOW,
        norm=NORM_YELLOW
    )
    sm_yellow.set_array([])

    sm_red = plt.cm.ScalarMappable(
        cmap=CMAP_RED,
        norm=NORM_RED
    )
    sm_red.set_array([])


    # =========================================================================
    # HORIZONTAL COLOR BARS
    # =========================================================================

    bar_left = 0.345
    bar_width = 0.310
    bar_height = 0.018

    yellow_bottom = 0.152
    red_bottom = 0.116

    cax_yellow = fig.add_axes([
        bar_left,
        yellow_bottom,
        bar_width,
        bar_height
    ])

    cbar_yellow = fig.colorbar(
        sm_yellow,
        cax=cax_yellow,
        orientation="horizontal",
        boundaries=CLASS_BOUNDS,
        ticks=CLASS_VALUES,
        spacing="uniform"
    )

    cbar_yellow.ax.set_xticklabels([
        "",
        "",
        "",
        "",
        ""
    ])

    cbar_yellow.ax.tick_params(
        length=0,
        width=0
    )

    cbar_yellow.outline.set_linewidth(
        0.45
    )


    cax_red = fig.add_axes([
        bar_left,
        red_bottom,
        bar_width,
        bar_height
    ])

    cbar_red = fig.colorbar(
        sm_red,
        cax=cax_red,
        orientation="horizontal",
        boundaries=CLASS_BOUNDS,
        ticks=CLASS_VALUES,
        spacing="uniform"
    )

    cbar_red.ax.set_xticklabels([
        "",
        "",
        "",
        "",
        ""
    ])

    cbar_red.ax.tick_params(
        length=0,
        width=0
    )

    cbar_red.outline.set_linewidth(
        0.45
    )


    # =========================================================================
    # CLASS LABELS
    # =========================================================================

    label_y = 0.075

    for i, label in enumerate(
        CLASS_LABELS
    ):

        x = (
            bar_left
            + (i + 0.5) / 5.0 * bar_width
        )

        fig.text(
            x,
            label_y,
            label,
            ha="center",
            va="center",
            fontsize=6.2,
            family="sans-serif",
            color="black"
        )


    # =========================================================================
    # COLOR SCALE TITLE
    # =========================================================================

    fig.text(
        bar_left + bar_width / 2,
        0.035,
        "Probability",
        ha="center",
        va="center",
        fontsize=7.8,
        fontweight="normal",
        family="sans-serif",
        color="black"
    )


    # =========================================================================
    # SAVE — 600 DPI
    # =========================================================================

    os.makedirs(
        os.path.dirname(
            OUTPUT_FILE
        ),
        exist_ok=True
    )

    fig.savefig(
        OUTPUT_FILE,
        dpi=600,
        bbox_inches="tight",
        pad_inches=0.025,
        facecolor="white",
        edgecolor="none"
    )

    plt.show()

    plt.close(fig)

    print(
        f"Saved: {OUTPUT_FILE}"
    )


# =============================================================================
# RUN
# =============================================================================

plot_zoom_p3risk_maps()

